# MVP — Clusterização de clientes e estratégia comercial
**Universidade de Brasília · Departamento de Engenharia de Produção · Prof. André Luiz Marques Serrano**
Trabalho individual (10,0 pts) · Pipeline de dados em nuvem (Databricks) · Base: *Customer Personality Analysis* (Kaggle)

Este notebook do Colab **prepara e dispara** o MVP: coleta e documenta os dados, analisa a qualidade, define o modelo estrela, gera o catálogo de dados, **gera os notebooks Databricks** (Bronze → Silver → Gold → Análise), ajuda a levá-los ao Databricks e monta o repositório do GitHub (README, catálogo, evidências).

### Como usar
1. Execute as células em ordem (Ambiente → Executar tudo). Só a célula de configuração (seção 3) e as marcadas `# TODO: preencha` pedem edição.
2. Ao final, baixe o `.zip` (seção 15) e suba o conteúdo no seu repositório público do GitHub.
3. Leve os notebooks gerados ao Databricks (seção 10) e execute o job; depois complete as discussões `[preencher ...]` com os números reais.

### Premissas deste trabalho (declaradas)
- **Nenhum resultado é inventado:** toda saída numérica vem da execução. Nas discussões há placeholders `[preencher ...]`.
- Plataforma: Databricks **Free Edition** (computação serverless; Unity Catalog com catálogo `workspace`). Se o seu workspace for diferente, ajuste `CATALOG` na seção 3.
- O volume é pequeno (2.240 linhas): Silver e clusterização usam pandas/scikit-learn dentro do Databricks; Bronze, escrita Delta, esquema estrela e as visões analíticas usam Spark/SQL.
- Clusterização: K-Means sobre RFM + perfil; `k` é escolhido pelo maior *silhouette* com `k ≥ 3` (k=2 só separa "alto" de "baixo" valor, pouco acionável) — decisão declarada e discutida.
- `Z_CostContact` e `Z_Revenue` são **premissa** de custo por contato e receita por aceite (a base não documenta o significado).

## 1. Mapa do checklist de entrega (seção 8 do PDF) e da rubrica (seção 7)

| Item do checklist / critério (pontos) | Onde está | Evidência no repositório |
|---|---|---|
| Objetivo e perguntas documentados e íntegros (**Objetivo 1,0**) | Seção 2 deste notebook; README | `README.md` |
| Origem, licença e data de coleta registradas (**Coleta 0,5**) | Seções 2 e 4 | `catalogo/metadados_coleta.json` |
| Dados persistidos na nuvem (**Coleta/Carga**) | Seção 10; notebooks 01–03 | prints das tabelas Delta em `evidencias/` |
| Modelo de dados descrito + Catálogo com domínios e linhagem (**Modelagem 2,0**) | Seções 7 e 8 | `catalogo/catalogo_dados.csv/.md`, `catalogo/diagrama_estrela.*` |
| Transformação e carga documentadas, regras explícitas (**Carga 1,0**) | Seção 6; notebooks 01–03; job | `catalogo/regras_transformacao.csv`, `notebooks/` |
| Análise de qualidade de todos os atributos (**Análise 1,0**) | Seção 5 | `evidencias/qualidade_*.png`, tabela no README |
| Cada pergunta com resposta, evidência e discussão (**Análise 1,0 + 1,0**) | Notebook Databricks `04_analise_p1_p5` | prints dos gráficos em `evidencias/` |
| Discussão geral conectando respostas à solução (**Análise**) | `04_analise_p1_p5` (última seção) | README |
| Autoavaliação presente (**Autoavaliação 0,5**) | Seção 12 | README |
| Evidências legíveis e referenciadas no texto (**Capricho 2,0**) | Seção 14 | `evidencias/` |
| Repositório público e conferido em janela anônima (**Capricho**) | Seção 13 | link na postagem do fórum |
| Ética, licença e LGPD (seção 4 do PDF) | Seção 11 | README |

## 2. Objetivo, perguntas de negócio e origem dos dados

**Problema.** A empresa trata a base de clientes de forma homogênea.
**Hipótese.** Existem segmentos naturais com valor, comportamento e resposta a campanhas diferentes; priorizar o orçamento comercial por segmento aumenta o retorno.

**Perguntas de negócio** (mantidas integralmente, mesmo que alguma não seja respondida):

- **P1.** Quantos segmentos naturais de clientes existem (K-Means sobre RFM + perfil demográfico) e qual o perfil de cada um em renda, composição familiar, gasto e frequência?
- **P2.** Qual a contribuição de cada segmento para a receita total e o gasto médio por cliente? Que fração dos clientes concentra 80% do gasto (Pareto)?
- **P3.** Como os segmentos diferem em mix de categorias (vinhos, carnes, peixes, frutas, doces, ouro) e em canal (loja, web, catálogo)? Qual o papel das visitas web e das compras com desconto?
- **P4.** Qual a taxa de aceitação das campanhas (1 a 5 e a última) por segmento? Sob a premissa declarada custo por contato = `Z_CostContact` e receita por aceite = `Z_Revenue`, qual segmento tem melhor retorno estimado? *(O significado de `Z_*` é uma premissa, pois a base não o documenta.)*
- **P5.** Quais clientes têm alto gasto histórico e recência alta (risco de inatividade) e que ação de retenção/reativação faz sentido para eles?

**Entrega final de negócio:** playbook comercial com uma ação recomendada por segmento (oferta, canal, prioridade).

**Origem dos dados.** Kaggle — *Customer Personality Analysis* (`imakash3011/customer-personality-analysis`), arquivo `marketing_campaign.csv`, separado por TAB, 2.240 linhas × 29 colunas.
**Licença.** CC0 (domínio público) conforme indicado na página do dataset — *confirme na página no dia da entrega* (`# TODO` na seção 3).
**Data de coleta, volume e formato:** registrados automaticamente na seção 4.

## 3. Configuração (única célula de parâmetros)

In [ ]:
# ============================ CONFIGURAÇÃO ÚNICA ============================
import os, sys, json, hashlib, datetime as dt, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# --- Pastas -----------------------------------------------------------------
BASE = (Path("/content/mvp_unb") if Path("/content").exists() else Path("mvp_unb")).resolve()
for sub in ["dados", "catalogo", "evidencias", "scripts", "notebooks", "databricks", "databricks/_templates"]:
    (BASE / sub).mkdir(parents=True, exist_ok=True)
os.chdir(BASE)

# --- Dados ------------------------------------------------------------------
KAGGLE_DATASET = "imakash3011/customer-personality-analysis"
CSV_NAME = "marketing_campaign.csv"
CSV_LOCAL_PATH = ""           # opcional: caminho de um CSV já existente (pula o download)
LICENCA_DADOS = "CC0: Public Domain"   # TODO: confirme na página do dataset e atualize se for diferente
DATA_CONFERENCIA_LICENCA = "AAAA-MM-DD"  # TODO: preencha a data em que conferiu a licença

# --- Databricks (Unity Catalog; Free Edition usa o catálogo "workspace") -----
CATALOG = "workspace"
SCHEMA = "mvp_marketing"
VOLUME = "raw"
USAR_SDK = False   # True = tenta enviar CSV/notebooks e criar o job via Databricks SDK (seção 10); False = passo manual

# --- Parâmetros do modelo ---------------------------------------------------
SEED = 42
K_MAX = 8                 # testa k de 2 até K_MAX
K_MIN_ESCOLHA = 3         # k escolhido = maior silhouette com k >= K_MIN_ESCOLHA
PCT_RISCO_GASTO = 0.75    # P5: alto gasto = acima deste percentil
PCT_RISCO_RECENCIA = 0.75 # P5: recência alta (dias sem comprar) = acima deste percentil

# --- Autor (usado no README/LICENSE) ----------------------------------------
NOME_ALUNO = "TODO: preencha seu nome"
MATRICULA = "TODO: preencha"
print("Pasta de trabalho:", BASE.resolve())

In [ ]:
%pip install -q kagglehub databricks-sdk

## 4. Coleta
Ordem de tentativa: **(1)** arquivo já existente em `dados/` ou em `CSV_LOCAL_PATH`; **(2)** download via `kagglehub` (dataset público, normalmente sem login); **(3)** API do Kaggle com `kaggle.json`; **(4)** upload manual do CSV.

*Para o passo 3:* Kaggle → *Settings → API → Create New Token* baixa o `kaggle.json`; envie-o pelo painel de arquivos do Colab para `~/.kaggle/kaggle.json` (ou defina as variáveis `KAGGLE_USERNAME` / `KAGGLE_KEY`).

O script registra automaticamente data, linhas/colunas, tamanho e hash **sha256** (reprodutibilidade, seção 3.2 do PDF).

In [ ]:
import glob, shutil
import pandas as pd

destino = BASE / "dados" / CSV_NAME
fonte = None

def _copiar(origem):
    shutil.copy(origem, destino)

# (1) arquivo já existente
if CSV_LOCAL_PATH and Path(CSV_LOCAL_PATH).exists():
    _copiar(CSV_LOCAL_PATH); fonte = f"arquivo local informado ({CSV_LOCAL_PATH})"
elif destino.exists():
    fonte = "arquivo já existente em dados/ (reaproveitado)"

# (2) kagglehub
if fonte is None:
    try:
        import kagglehub
        pasta = kagglehub.dataset_download(KAGGLE_DATASET)
        achados = glob.glob(f"{pasta}/**/{CSV_NAME}", recursive=True)
        if not achados:
            raise FileNotFoundError(f"{CSV_NAME} não está em {pasta}")
        _copiar(achados[0]); fonte = f"kagglehub ({KAGGLE_DATASET})"
    except Exception as e:
        print("kagglehub indisponível:", repr(e)[:200])

# (3) API do Kaggle com kaggle.json / variáveis de ambiente
if fonte is None:
    try:
        tem_cred = (Path.home() / ".kaggle" / "kaggle.json").exists() or ("KAGGLE_KEY" in os.environ)
        if not tem_cred:
            raise RuntimeError("sem credenciais do Kaggle (kaggle.json)")
        os.system(f"kaggle datasets download -d {KAGGLE_DATASET} -p dados --unzip -q")
        if not destino.exists():
            raise FileNotFoundError("download pela API não gerou o CSV")
        fonte = f"Kaggle API ({KAGGLE_DATASET})"
    except Exception as e:
        print("API do Kaggle indisponível:", repr(e)[:200])

# (4) upload manual
if fonte is None:
    try:
        from google.colab import files
        print("Selecione o arquivo marketing_campaign.csv no seu computador:")
        enviados = files.upload()
        nome = next(iter(enviados))
        shutil.move(nome, destino); fonte = "upload manual"
    except ImportError:
        raise SystemExit("Não foi possível obter o CSV. Coloque-o em dados/marketing_campaign.csv e rode novamente.")

# --- Leitura (TAB) com verificação do separador ---
raw = pd.read_csv(destino, sep="\t")
if raw.shape[1] == 1:   # separador inesperado: tenta detectar
    raw = pd.read_csv(destino, sep=None, engine="python")

# --- Metadados de coleta ---
sha = hashlib.sha256(destino.read_bytes()).hexdigest()
meta = {
    "fonte": fonte, "dataset": KAGGLE_DATASET, "arquivo": CSV_NAME, "licenca": LICENCA_DADOS,
    "data_conferencia_licenca": DATA_CONFERENCIA_LICENCA,
    "data_coleta_utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
    "linhas": int(raw.shape[0]), "colunas": int(raw.shape[1]),
    "tamanho_bytes": destino.stat().st_size, "sha256": sha, "formato": "CSV separado por TAB, UTF-8",
}
(BASE / "catalogo" / "metadados_coleta.json").write_text(json.dumps(meta, indent=2, ensure_ascii=False))
print(json.dumps(meta, indent=2, ensure_ascii=False))
if raw.shape != (2240, 29):
    print(f"\nATENÇÃO: esperado (2240, 29), obtido {raw.shape}. Verifique se é a mesma versão do dataset.")
raw.head()

## 5. Análise de qualidade de dados (por atributo)
Dimensões examinadas: **completude** (nulos), **unicidade** (duplicatas), **consistência** (contradições entre atributos), **conformidade** (aderência ao domínio), **acurácia** (valores implausíveis) e **atualidade**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "font.size": 10})

def perfil_atributos(df):
    """Tabela com uma linha por atributo: tipo, nulos, únicos, mín/máx, outliers (IQR 1,5) e observações."""
    linhas = []
    for c in df.columns:
        s = df[c]; num = pd.api.types.is_numeric_dtype(s)
        obs = []
        if s.nunique(dropna=False) == 1: obs.append("constante")
        out = np.nan
        if num and s.nunique() > 2:
            q1, q3 = s.quantile([.25, .75]); iqr = q3 - q1
            out = int(((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum())
        linhas.append({"atributo": c, "tipo": str(s.dtype), "nulos": int(s.isna().sum()),
                       "% nulos": round(100 * s.isna().mean(), 2), "únicos": int(s.nunique()),
                       "mín": s.min() if num else "", "máx": s.max() if num else "",
                       "outliers_IQR": out, "obs": ", ".join(obs)})
    return pd.DataFrame(linhas)

perfil = perfil_atributos(raw)
perfil.to_csv(BASE / "catalogo" / "qualidade_por_atributo.csv", index=False)
pd.set_option("display.width", 200, "display.max_rows", 100)
perfil

In [ ]:
# ---- Testes por dimensão de qualidade ----
datas = pd.to_datetime(raw["Dt_Customer"], format="%d-%m-%Y", errors="coerce")
gastos = ["MntWines", "MntFruits", "MntMeatProducts", "MntFishProducts", "MntSweetProducts", "MntGoldProds"]
canais = ["NumWebPurchases", "NumCatalogPurchases", "NumStorePurchases"]
compras = raw[canais].sum(axis=1)
idade_ref = datas.max().year - raw["Year_Birth"]
q1, q3 = raw["Income"].quantile([.25, .75])
binarias = ["AcceptedCmp1", "AcceptedCmp2", "AcceptedCmp3", "AcceptedCmp4", "AcceptedCmp5", "Complain", "Response"]

testes = [
    ("Completude", "Income", "valores nulos", int(raw["Income"].isna().sum()), "Imputar pela mediana do nível de educação (R3)"),
    ("Completude", "demais colunas", "nulos fora de Income", int(raw.drop(columns="Income").isna().sum().sum()), "Nenhuma ação"),
    ("Unicidade", "linha inteira", "linhas duplicadas", int(raw.duplicated().sum()), "Nenhuma ação"),
    ("Unicidade", "ID", "IDs duplicados", int(raw["ID"].duplicated().sum()), "Nenhuma ação"),
    ("Consistência", "NumDealsPurchases vs canais", "compras com desconto > compras totais", int((raw["NumDealsPurchases"] > compras).sum()), "Manter e limitar % a 1 (R9)"),
    ("Consistência", "compras", "clientes com 0 compras mas gasto > 0", int(((compras == 0) & (raw[gastos].sum(axis=1) > 0)).sum()), "Manter; ticket/shares = 0 (R10)"),
    ("Conformidade", "Dt_Customer", "datas fora do formato dd-mm-aaaa", int(datas.isna().sum()), "Conversão explícita (R2)"),
    ("Conformidade", "Marital_Status", "categorias fora do domínio esperado (Alone/Absurd/YOLO)", int(raw["Marital_Status"].isin(["Alone", "Absurd", "YOLO"]).sum()), "Recodificar (R6)"),
    ("Conformidade", "Education", "categoria '2n Cycle' (nomenclatura distinta)", int((raw["Education"] == "2n Cycle").sum()), "Padronizar como Mestrado, preservando original (R7)"),
    ("Conformidade", "binárias", "valores fora de {0,1}", int(sum((~raw[c].isin([0, 1])).sum() for c in binarias)), "Nenhuma ação"),
    ("Conformidade", "Z_CostContact, Z_Revenue", "colunas constantes", int(sum(raw[c].nunique() == 1 for c in ["Z_CostContact", "Z_Revenue"])), "Descartar; usar só como premissa de P4 (R1)"),
    ("Acurácia", "Year_Birth", "idade > 100 anos na data de referência", int((idade_ref > 100).sum()), "Imputar idade pela mediana (R5)"),
    ("Acurácia", "Income", "outlier extremo (> Q3+3·IQR)", int((raw["Income"] > q3 + 3 * (q3 - q1)).sum()), "Tratar como erro e imputar (R4)"),
    ("Atualidade", "Dt_Customer", f"período dos cadastros: {datas.min().date()} a {datas.max().date()}", int((pd.Timestamp.today().year - datas.max().year)), "Dados defasados (anos desde o último cadastro); registrar como limitação"),
]
qualidade = pd.DataFrame(testes, columns=["dimensão", "atributo", "teste", "ocorrências", "decisão"])
qualidade.to_csv(BASE / "catalogo" / "qualidade_testes.csv", index=False)
qualidade

In [ ]:
# ---- Confirmação dos achados previamente verificados (devem bater com a execução) ----
vinho_share = raw["MntWines"].sum() / raw[gastos].sum().sum()
loja_share = raw["NumStorePurchases"].sum() / compras.sum()
achados = [
    ("Income tem 24 nulos", 24, int(raw["Income"].isna().sum())),
    ("Income máximo = 666.666", 666666, raw["Income"].max()),
    ("Year_Birth mínimo = 1893", 1893, raw["Year_Birth"].min()),
    ("Marital_Status contém Alone, Absurd e YOLO", True, {"Alone", "Absurd", "YOLO"} <= set(raw["Marital_Status"])),
    ("Education contém '2n Cycle'", True, "2n Cycle" in set(raw["Education"])),
    ("Z_CostContact constante = 3", 3, raw["Z_CostContact"].unique().tolist() == [3] and 3),
    ("Z_Revenue constante = 11", 11, raw["Z_Revenue"].unique().tolist() == [11] and 11),
    ("Sem duplicatas por linha", 0, int(raw.duplicated().sum())),
    ("Sem duplicatas por ID", 0, int(raw["ID"].duplicated().sum())),
    ("Response ≈ 14,9%", 0.149, round(raw["Response"].mean(), 3)),
    ("Vinhos ≈ 50% do gasto", 0.50, round(vinho_share, 3)),
    ("Loja física ≈ 46% das compras", 0.46, round(loja_share, 3)),
]
conf = pd.DataFrame([{"achado": a, "esperado": e, "observado": o, "confere": (abs(o - e) < 0.006) if isinstance(e, float) else (o == e)} for a, e, o in achados])
display(conf)
if not conf["confere"].all():
    print("ATENÇÃO: há achados que não conferem — revise a versão do dataset antes de prosseguir.")

# ---- Gráficos de evidência da qualidade ----
fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
ax[0].boxplot(raw["Income"].dropna(), vert=False); ax[0].set_title("Income (outlier em 666.666)")
ax[1].hist(raw["Year_Birth"], bins=30, color="#2a6f97"); ax[1].set_title("Year_Birth (anos implausíveis < 1900)")
raw["Marital_Status"].value_counts().plot.barh(ax=ax[2], color="#e07a1f"); ax[2].set_title("Marital_Status (categorias estranhas)")
plt.tight_layout(); plt.savefig(BASE / "evidencias" / "qualidade_outliers_categorias.png", bbox_inches="tight"); plt.show()

## 6. Tratamento: decisões justificadas e regras de negócio

| Problema | Decisão | Justificativa |
|---|---|---|
| `Income` nulo (24) | Imputar pela **mediana de renda do mesmo nível de educação** | Renda varia com escolaridade; a mediana é robusta a outliers. Flag `flag_renda_imputada` preserva a rastreabilidade. |
| `Income` = 666.666 (outlier) | Tratar como erro de digitação (> Q3 + 3·IQR) → nulo → imputar | É ~4× o segundo maior valor; manter distorceria média, escala e K-Means. Winsorizar manteria um valor sem sentido. |
| `Year_Birth` < 1900 | Idade implausível (> 100) → imputar mediana | Poucos casos; remover perderia o cliente. Flag `flag_idade_corrigida`. |
| Idade e tempo de cliente | Calcular em relação à **maior `Dt_Customer`** | Os dados são de 2012–2014; usar a data de hoje inflaria a idade de todos. |
| `Marital_Status` Alone/Absurd/YOLO | Alone → Solteiro(a); Absurd e YOLO → "Não informado" | Evita inventar valor; preserva a coluna original. |
| `Education` "2n Cycle" | Padronizar como Mestrado (premissa), mantendo o original | 2º ciclo (Bologna) ≈ mestrado. |
| `Z_CostContact`, `Z_Revenue` | Descartar da Silver; usar como **premissa** de P4 | Constantes: sem poder discriminante. |
| `compras_desconto` > compras totais (3) | Manter; limitar % a 1 | Inconsistência pequena, registrada. |

A lógica fica em **funções reutilizadas no Colab e no Databricks** (arquivo `scripts/mvp_core.py`, gerado abaixo), para garantir que o ensaio local e a execução na nuvem façam exatamente o mesmo.

In [ ]:
%%writefile scripts/mvp_core.py
# ============================================================
# NÚCLEO DE LÓGICA DO MVP (usado no Colab e nos notebooks Databricks)
# Funções puras em pandas/sklearn: Silver (limpeza + features),
# clusterização (K-Means) e construção das tabelas do esquema estrela.
# Volume de dados pequeno (2.240 linhas): pandas é suficiente e testável.
# ============================================================
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, adjusted_rand_score

# ---------- Mapeamentos e nomes -------------------------------------------
RENOMEAR = {
    "ID": "cliente_id", "Year_Birth": "ano_nascimento", "Education": "educacao_original",
    "Marital_Status": "estado_civil_original", "Income": "renda",
    "Kidhome": "filhos_criancas", "Teenhome": "filhos_adolescentes",
    "Dt_Customer": "data_cadastro", "Recency": "recencia_dias",
    "MntWines": "gasto_vinhos", "MntFruits": "gasto_frutas", "MntMeatProducts": "gasto_carnes",
    "MntFishProducts": "gasto_peixes", "MntSweetProducts": "gasto_doces", "MntGoldProds": "gasto_ouro",
    "NumDealsPurchases": "compras_desconto", "NumWebPurchases": "compras_web",
    "NumCatalogPurchases": "compras_catalogo", "NumStorePurchases": "compras_loja",
    "NumWebVisitsMonth": "visitas_web_mes",
    "AcceptedCmp1": "campanha_1", "AcceptedCmp2": "campanha_2", "AcceptedCmp3": "campanha_3",
    "AcceptedCmp4": "campanha_4", "AcceptedCmp5": "campanha_5", "Response": "campanha_ultima",
    "Complain": "reclamacao_2anos",
}
CATEGORIAS = ["vinhos", "carnes", "peixes", "frutas", "doces", "ouro"]
COLS_GASTO = [f"gasto_{c}" for c in CATEGORIAS]
COLS_CAMPANHA = ["campanha_1", "campanha_2", "campanha_3", "campanha_4", "campanha_5", "campanha_ultima"]
MAP_EDUCACAO = {  # original -> (padronizada, nível ordinal)
    "Basic": ("Básico", 1), "Graduation": ("Graduação", 2),
    "2n Cycle": ("Mestrado", 3),  # premissa: 2n Cycle = 2º ciclo (Bologna) ~ mestrado
    "Master": ("Mestrado", 3), "PhD": ("Doutorado", 4),
}
MAP_ESTADO_CIVIL = {  # original -> (padronizado, grupo)
    "Married": ("Casado(a)", "Com parceiro(a)"), "Together": ("União estável", "Com parceiro(a)"),
    "Single": ("Solteiro(a)", "Sem parceiro(a)"), "Alone": ("Solteiro(a)", "Sem parceiro(a)"),
    "Divorced": ("Divorciado(a)", "Sem parceiro(a)"), "Widow": ("Viúvo(a)", "Sem parceiro(a)"),
    "Absurd": ("Não informado", "Não informado"), "YOLO": ("Não informado", "Não informado"),
}
FEATURES_CLUSTER = ["recencia_dias", "compras_total", "gasto_total_log", "renda", "idade", "n_filhos"]


def _num(df, cols):
    for c in cols:
        df[c] = pd.to_numeric(df[c], errors="raise")
    return df


# ---------- SILVER ---------------------------------------------------------
def silver_transform(raw):
    """Recebe o Bronze (pandas) e devolve (silver, log).
    log = lista de dicts {regra, afetados, decisao} para documentar cada regra de negócio."""
    log = []
    df = raw[[c for c in raw.columns if not c.startswith("_")]].copy()  # remove metadados de ingestão
    textos = ["Education", "Marital_Status", "Dt_Customer"]
    df = _num(df, [c for c in df.columns if c not in textos])

    # R1. Colunas constantes não carregam informação -> descartadas (valores guardados como premissa de P4)
    constantes = {c: df[c].iloc[0] for c in df.columns if df[c].nunique(dropna=False) == 1}
    df = df.drop(columns=list(constantes))
    log.append({"regra": "R1 Colunas constantes", "afetados": len(constantes),
                "decisao": f"Descartadas {list(constantes)} (valores {list(constantes.values())}); "
                           "mantidos só como premissa de custo/receita em P4"})

    df = df.rename(columns=RENOMEAR)

    # R2. Datas: Dt_Customer vem como dd-mm-aaaa
    df["data_cadastro"] = pd.to_datetime(df["data_cadastro"], format="%d-%m-%Y", errors="raise")
    data_ref = df["data_cadastro"].max()  # idade/tempo calculados na data de referência da base, não hoje
    log.append({"regra": "R2 Data de referência", "afetados": len(df),
                "decisao": f"data_ref = max(Dt_Customer) = {data_ref.date()}; idade e tempo de cliente calculados nela"})

    # R3. Renda: nulos + outlier extremo (> Q3 + 3*IQR) -> NaN -> imputação pela mediana do nível de educação
    q1, q3 = df["renda"].quantile([.25, .75])
    limite = q3 + 3 * (q3 - q1)
    m_out = df["renda"] > limite
    m_nulo = df["renda"].isna()
    df["flag_renda_outlier"] = m_out
    df["flag_renda_imputada"] = m_out | m_nulo
    base = df.loc[~(m_out | m_nulo)]
    med_educ = base.groupby("educacao_original")["renda"].median()
    df.loc[m_out, "renda"] = np.nan
    df["renda"] = df["renda"].fillna(df["educacao_original"].map(med_educ)).fillna(base["renda"].median())
    log.append({"regra": "R3 Renda nula", "afetados": int(m_nulo.sum()),
                "decisao": "Imputada pela mediana de renda do mesmo nível de educação (flag_renda_imputada)"})
    log.append({"regra": "R4 Renda outlier extremo", "afetados": int(m_out.sum()),
                "decisao": f"Valores > {limite:,.0f} (Q3+3·IQR) tratados como erro de digitação: viram nulo e são imputados"})

    # R5. Idade implausível
    df["idade"] = data_ref.year - df["ano_nascimento"]
    m_idade = (df["idade"] > 100) | (df["idade"] < 16)
    df["flag_idade_corrigida"] = m_idade
    df.loc[m_idade, "idade"] = np.nan
    df["idade"] = df["idade"].fillna(df["idade"].median()).astype(int)
    df["ano_nascimento"] = (data_ref.year - df["idade"]).astype(int)
    log.append({"regra": "R5 Idade implausível", "afetados": int(m_idade.sum()),
                "decisao": "Idade > 100 ou < 16 na data de referência vira nulo e é imputada pela mediana (flag_idade_corrigida)"})

    # R6. Padronização de categorias
    desconhecidas = set(df["educacao_original"]) - set(MAP_EDUCACAO)
    desconhecidas |= set(df["estado_civil_original"]) - set(MAP_ESTADO_CIVIL)
    if desconhecidas:
        raise ValueError(f"Categorias não mapeadas: {desconhecidas}. Atualize MAP_EDUCACAO / MAP_ESTADO_CIVIL.")
    df["educacao"] = df["educacao_original"].map(lambda x: MAP_EDUCACAO[x][0])
    df["educacao_nivel"] = df["educacao_original"].map(lambda x: MAP_EDUCACAO[x][1])
    df["estado_civil"] = df["estado_civil_original"].map(lambda x: MAP_ESTADO_CIVIL[x][0])
    df["estado_civil_grupo"] = df["estado_civil_original"].map(lambda x: MAP_ESTADO_CIVIL[x][1])
    n_est = int(df["estado_civil_original"].isin(["Alone", "Absurd", "YOLO"]).sum())
    log.append({"regra": "R6 Estado civil", "afetados": n_est,
                "decisao": "Alone -> Solteiro(a); Absurd e YOLO -> 'Não informado' (mantidos, sem inferir valor)"})
    log.append({"regra": "R7 Educação '2n Cycle'", "afetados": int((df["educacao_original"] == "2n Cycle").sum()),
                "decisao": "Premissa: 2º ciclo (Bologna) equivale a Mestrado; coluna original preservada"})

    # R8. Features derivadas
    df["gasto_total"] = df[COLS_GASTO].sum(axis=1)
    df["compras_total"] = df[["compras_web", "compras_catalogo", "compras_loja"]].sum(axis=1)  # canais; desconto é subconjunto
    df["n_filhos"] = df["filhos_criancas"] + df["filhos_adolescentes"]
    df["tempo_cliente_dias"] = (data_ref - df["data_cadastro"]).dt.days
    df["flag_sem_compras"] = df["compras_total"] == 0
    comp = df["compras_total"].replace(0, np.nan)
    df["ticket_medio"] = (df["gasto_total"] / comp).fillna(0.0)
    for canal in ["web", "catalogo", "loja"]:
        df[f"share_{canal}"] = (df[f"compras_{canal}"] / comp).fillna(0.0)
    df["pct_compras_desconto"] = (df["compras_desconto"] / comp).clip(upper=1).fillna(0.0)
    for c in CATEGORIAS:
        df[f"pct_gasto_{c}"] = df[f"gasto_{c}"] / df["gasto_total"].replace(0, np.nan)
        df[f"pct_gasto_{c}"] = df[f"pct_gasto_{c}"].fillna(0.0)
    df["n_campanhas_aceitas"] = df[COLS_CAMPANHA[:5]].sum(axis=1)  # campanhas 1 a 5
    log.append({"regra": "R8 Features", "afetados": len(df),
                "decisao": "gasto_total, compras_total (web+catálogo+loja; desconto é subconjunto), n_filhos, "
                           "tempo_cliente_dias, ticket_medio, share por canal, pct por categoria, n_campanhas_aceitas"})
    n_desc = int((df["compras_desconto"] > df["compras_total"]).sum())
    log.append({"regra": "R9 Consistência: compras com desconto > compras totais", "afetados": n_desc,
                "decisao": "Mantidos; pct_compras_desconto limitado a 1 (inconsistência registrada, não corrigida)"})
    log.append({"regra": "R10 Clientes sem compras", "afetados": int(df["flag_sem_compras"].sum()),
                "decisao": "Mantidos (têm gasto > 0); ticket médio e shares ficam 0 (flag_sem_compras)"})

    ordem = ["cliente_id", "ano_nascimento", "idade", "data_cadastro", "educacao_original", "educacao",
             "educacao_nivel", "estado_civil_original", "estado_civil", "estado_civil_grupo", "renda",
             "filhos_criancas", "filhos_adolescentes", "n_filhos", "recencia_dias", "tempo_cliente_dias"]
    resto = [c for c in df.columns if c not in ordem]
    df = df[ordem + resto].reset_index(drop=True)
    for c in ["flag_renda_outlier", "flag_renda_imputada", "flag_idade_corrigida", "flag_sem_compras"]:
        df[c] = df[c].astype(bool)
    return df, pd.DataFrame(log), {k: float(v) for k, v in constantes.items()}


# ---------- CLUSTERIZAÇÃO ---------------------------------------------------
def matriz_cluster(silver):
    X = silver.copy()
    X["gasto_total_log"] = np.log1p(X["gasto_total"])  # reduz assimetria do gasto
    Z = StandardScaler().fit_transform(X[FEATURES_CLUSTER])
    return Z


def clusterizar(silver, k_plot_min=2, k_max=8, k_min_escolha=3, seed=42, seeds_estab=(1, 7, 21, 42, 99)):
    """K-Means. Escolhe k pelo maior silhouette em [k_min_escolha, k_max]; mede estabilidade (ARI entre sementes)."""
    Z = matriz_cluster(silver)
    linhas = []
    for k in range(k_plot_min, k_max + 1):
        km = KMeans(n_clusters=k, n_init=20, random_state=seed).fit(Z)
        linhas.append({"k": k, "inercia": km.inertia_, "silhouette": silhouette_score(Z, km.labels_)})
    metricas = pd.DataFrame(linhas)
    elegiveis = metricas[metricas["k"] >= k_min_escolha]
    k = int(elegiveis.loc[elegiveis["silhouette"].idxmax(), "k"])
    base = KMeans(n_clusters=k, n_init=20, random_state=seed).fit(Z).labels_
    aris = []
    for s in seeds_estab:
        outro = KMeans(n_clusters=k, n_init=20, random_state=s).fit(Z).labels_
        aris.append(adjusted_rand_score(base, outro))
    estab = {"k": k, "ari_medio": float(np.mean(aris)), "ari_min": float(np.min(aris)),
             "sementes": list(seeds_estab), "silhouette_k": float(metricas.loc[metricas.k == k, "silhouette"].iloc[0])}
    # ordena segmentos por gasto médio decrescente: S1 = maior gasto
    tmp = pd.DataFrame({"lab": base, "gasto": silver["gasto_total"].values})
    ordem = tmp.groupby("lab")["gasto"].mean().sort_values(ascending=False).index.tolist()
    novo = {lab: i + 1 for i, lab in enumerate(ordem)}
    seg_id = pd.Series(base).map(novo).astype(int).values
    return seg_id, metricas, estab


def rotulo_segmento(i, k):
    if i == 1:
        return f"S{i} - maior gasto"
    if i == k:
        return f"S{i} - menor gasto"
    return f"S{i} - gasto intermediário"


# ---------- GOLD: esquema estrela -------------------------------------------
def construir_gold(silver, seg_id):
    s = silver.copy()
    s["segmento_id"] = seg_id
    k = int(s["segmento_id"].max())

    dim_educacao = (s[["educacao_original", "educacao", "educacao_nivel"]].drop_duplicates()
                    .sort_values(["educacao_nivel", "educacao_original"]).reset_index(drop=True))
    dim_educacao.insert(0, "educacao_id", range(1, len(dim_educacao) + 1))
    dim_estado = (s[["estado_civil", "estado_civil_grupo"]].drop_duplicates()
                  .sort_values("estado_civil").reset_index(drop=True))
    dim_estado.insert(0, "estado_civil_id", range(1, len(dim_estado) + 1))

    datas = pd.Series(pd.to_datetime(s["data_cadastro"].unique())).sort_values()
    dim_tempo = pd.DataFrame({"data": datas})
    dim_tempo.insert(0, "data_id", dim_tempo["data"].dt.strftime("%Y%m%d").astype(int))
    dim_tempo["ano"] = dim_tempo["data"].dt.year
    dim_tempo["trimestre"] = dim_tempo["data"].dt.quarter
    dim_tempo["mes"] = dim_tempo["data"].dt.month
    dim_tempo["dia_semana"] = dim_tempo["data"].dt.dayofweek + 1  # 1=segunda
    dim_tempo["ano_mes"] = dim_tempo["data"].dt.strftime("%Y-%m")

    dim_cliente = s[["cliente_id", "ano_nascimento", "idade", "renda", "n_filhos", "filhos_criancas",
                     "filhos_adolescentes", "flag_renda_outlier", "flag_renda_imputada",
                     "flag_idade_corrigida"]].copy()
    dim_cliente["faixa_etaria"] = pd.cut(dim_cliente["idade"], [0, 34, 44, 54, 64, 200],
                                         labels=["até 34", "35-44", "45-54", "55-64", "65+"]).astype(str)
    dim_cliente["faixa_renda"] = pd.qcut(dim_cliente["renda"], 4, labels=["Q1 (menor)", "Q2", "Q3", "Q4 (maior)"]).astype(str)
    dim_cliente["tem_filhos"] = dim_cliente["n_filhos"] > 0

    agg = s.groupby("segmento_id").agg(n_clientes=("cliente_id", "count"), gasto_medio=("gasto_total", "mean"),
                                       renda_media=("renda", "mean"), recencia_media=("recencia_dias", "mean")).reset_index()
    agg["rotulo"] = [rotulo_segmento(i, k) for i in agg["segmento_id"]]
    dim_segmento = agg[["segmento_id", "rotulo", "n_clientes", "gasto_medio", "renda_media", "recencia_media"]]

    fato = s.merge(dim_educacao[["educacao_id", "educacao_original"]], on="educacao_original") \
            .merge(dim_estado[["estado_civil_id", "estado_civil"]], on="estado_civil")
    fato["data_id"] = fato["data_cadastro"].dt.strftime("%Y%m%d").astype(int)
    cols_fato = (["cliente_id", "educacao_id", "estado_civil_id", "data_id", "segmento_id", "recencia_dias",
                  "tempo_cliente_dias", "gasto_total"] + COLS_GASTO +
                 [f"pct_gasto_{c}" for c in CATEGORIAS] +
                 ["compras_total", "compras_web", "compras_catalogo", "compras_loja", "compras_desconto",
                  "visitas_web_mes", "ticket_medio", "share_web", "share_catalogo", "share_loja",
                  "pct_compras_desconto"] + COLS_CAMPANHA + ["n_campanhas_aceitas", "reclamacao_2anos", "flag_sem_compras"])
    fato = fato[cols_fato].sort_values("cliente_id").reset_index(drop=True)
    assert len(fato) == len(s), "O join da fato duplicou ou perdeu clientes"
    return {"dim_cliente": dim_cliente, "dim_educacao": dim_educacao, "dim_estado_civil": dim_estado,
            "dim_tempo": dim_tempo, "dim_segmento": dim_segmento, "fato_cliente_vendas": fato}

In [ ]:
sys.path.insert(0, str(BASE / "scripts"))
import importlib, mvp_core
importlib.reload(mvp_core)
from mvp_core import *

# Ensaio local: roda a Silver sobre o CSV bruto (lido como texto, como na Bronze)
raw_txt = pd.read_csv(destino, sep="\t", dtype=str)
silver, log_regras, constantes = silver_transform(raw_txt)
log_regras.to_csv(BASE / "catalogo" / "regras_transformacao.csv", index=False)
print("Silver local:", silver.shape, "| nulos restantes:", int(silver.isna().sum().sum()))
print("Constantes descartadas (premissa de P4):", constantes)
display(log_regras)

## 7. Modelagem — Esquema Estrela
**Por que estrela (e não flat/snowflake):** as perguntas P1–P5 são análises de métricas (gasto, compras, campanhas) cortadas por atributos descritivos (segmento, escolaridade, estado civil, faixa de renda/idade, data de cadastro). O estrela dá joins simples e SQL legível, e a base é pequena demais para justificar a normalização do snowflake.

**Grão da fato:** uma linha por cliente (*snapshot* acumulado do período 2012–2014).

| Tabela | Tipo | Conteúdo |
|---|---|---|
| `fato_cliente_vendas` | fato | gasto por categoria e % do mix, compras por canal, desconto, visitas web, recência, campanhas, ticket, FKs |
| `dim_cliente` | dimensão | idade, faixa etária, renda, faixa de renda, filhos, flags de qualidade |
| `dim_educacao` | dimensão | nível original e padronizado |
| `dim_estado_civil` | dimensão | estado civil padronizado e grupo |
| `dim_tempo` | dimensão | data de cadastro (`Dt_Customer`): ano, trimestre, mês |
| `dim_segmento` | dimensão | resultado do cluster (rótulo, tamanho, médias) |

Abaixo roda-se o **ensaio local** da clusterização e da construção do esquema (os mesmos códigos serão executados no Databricks).

In [ ]:
# Ensaio local da clusterização (K-Means) e do esquema estrela
seg_id, metricas, estab = clusterizar(silver, k_max=K_MAX, k_min_escolha=K_MIN_ESCOLHA, seed=SEED)
gold = construir_gold(silver, seg_id)

print(f"k escolhido = {estab['k']} | silhouette = {estab['silhouette_k']:.3f}")
print(f"Estabilidade entre sementes {estab['sementes']}: ARI médio = {estab['ari_medio']:.3f} | mínimo = {estab['ari_min']:.3f}")
display(metricas.round(3))
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].plot(metricas["k"], metricas["inercia"], marker="o", color="#2a6f97"); ax[0].set_title("Cotovelo (inércia)")
ax[1].plot(metricas["k"], metricas["silhouette"], marker="o", color="#e07a1f"); ax[1].set_title("Silhouette")
for a in ax: a.axvline(estab["k"], ls="--", color="#999"); a.set_xlabel("k")
plt.tight_layout(); plt.savefig(BASE / "evidencias" / "ensaio_local_escolha_k.png", bbox_inches="tight"); plt.show()
for nome, df_ in gold.items():
    print(f"{nome:22s} {df_.shape}")

In [ ]:
# Diagrama do esquema estrela: mermaid (.md) e imagem (.png) em /catalogo
mermaid = """```mermaid
erDiagram
    FATO_CLIENTE_VENDAS }o--|| DIM_CLIENTE : cliente_id
    FATO_CLIENTE_VENDAS }o--|| DIM_EDUCACAO : educacao_id
    FATO_CLIENTE_VENDAS }o--|| DIM_ESTADO_CIVIL : estado_civil_id
    FATO_CLIENTE_VENDAS }o--|| DIM_TEMPO : data_id
    FATO_CLIENTE_VENDAS }o--|| DIM_SEGMENTO : segmento_id
    FATO_CLIENTE_VENDAS {
        int cliente_id PK
        int educacao_id FK
        int estado_civil_id FK
        int data_id FK
        int segmento_id FK
        double gasto_total
        double gasto_vinhos_carnes_peixes_frutas_doces_ouro
        int compras_web_catalogo_loja_desconto
        int recencia_dias
        int campanha_1_a_5_ultima
    }
    DIM_CLIENTE { int cliente_id PK
        int idade
        double renda
        string faixa_etaria
        string faixa_renda }
    DIM_EDUCACAO { int educacao_id PK
        string educacao
        int educacao_nivel }
    DIM_ESTADO_CIVIL { int estado_civil_id PK
        string estado_civil
        string estado_civil_grupo }
    DIM_TEMPO { int data_id PK
        date data
        int ano
        int trimestre }
    DIM_SEGMENTO { int segmento_id PK
        string rotulo
        int n_clientes }
```
"""
(BASE / "catalogo" / "diagrama_estrela.md").write_text("# Diagrama do esquema estrela\n\n" + mermaid)

from matplotlib.patches import FancyBboxPatch
fig, ax = plt.subplots(figsize=(10, 6)); ax.axis("off"); ax.set_xlim(0, 10); ax.set_ylim(0, 6)
def caixa(x, y, w, h, titulo, linhas, cor):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.05", fc=cor, ec="#333"))
    ax.text(x + w / 2, y + h - 0.25, titulo, ha="center", va="top", weight="bold", fontsize=9)
    ax.text(x + 0.15, y + h - 0.6, "\n".join(linhas), ha="left", va="top", fontsize=7.5)
caixa(3.6, 2.0, 2.8, 2.2, "fato_cliente_vendas", ["PK cliente_id", "FK educacao_id, estado_civil_id,", "   data_id, segmento_id", "gasto_total, gasto_<categoria> (6)", "compras_<canal>, compras_desconto", "recencia_dias, campanha_1..5/última"], "#cfe3f1")
caixa(0.2, 4.2, 2.6, 1.4, "dim_cliente", ["PK cliente_id", "idade, faixa_etaria", "renda, faixa_renda, n_filhos"], "#f6e2c8")
caixa(3.7, 4.5, 2.6, 1.2, "dim_segmento", ["PK segmento_id", "rotulo, n_clientes, médias"], "#f6e2c8")
caixa(7.2, 4.2, 2.6, 1.4, "dim_tempo", ["PK data_id", "data, ano, trimestre, mês"], "#f6e2c8")
caixa(0.2, 0.3, 2.6, 1.2, "dim_educacao", ["PK educacao_id", "educacao, educacao_nivel"], "#f6e2c8")
caixa(7.2, 0.3, 2.6, 1.2, "dim_estado_civil", ["PK estado_civil_id", "estado_civil, grupo"], "#f6e2c8")
for (x1, y1, x2, y2) in [(2.8, 4.7, 3.6, 3.6), (5.0, 4.5, 5.0, 4.2), (7.2, 4.7, 6.4, 3.6), (2.8, 1.2, 3.6, 2.4), (7.2, 1.2, 6.4, 2.4)]:
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle="->", color="#555"))
ax.set_title("Esquema estrela do MVP", weight="bold")
plt.savefig(BASE / "catalogo" / "diagrama_estrela.png", dpi=150, bbox_inches="tight"); plt.show()

## 8. Catálogo de dados (CSV e Markdown)
Campos exigidos pelo PDF (seção 3.3): nome, tipo, descrição, domínio (mín/máx ou categorias), obrigatoriedade (nulos e significado da ausência) e linhagem (origem, data de extração, transformações). Domínios e obrigatoriedade são **calculados dos dados**; descrições são texto de negócio.

In [ ]:
# ---- Descrições de negócio (descrição, transformação aplicada) ----
D = {}
# Colunas originais (conforme descrição do dataset no Kaggle)
D.update({
 "ID": ("Identificador sequencial do cliente (não é identificador direto)", "Original"),
 "Year_Birth": ("Ano de nascimento", "Original"), "Education": ("Nível de escolaridade", "Original"),
 "Marital_Status": ("Estado civil", "Original"), "Income": ("Renda anual do domicílio", "Original; 24 nulos"),
 "Kidhome": ("Nº de crianças no domicílio", "Original"), "Teenhome": ("Nº de adolescentes no domicílio", "Original"),
 "Dt_Customer": ("Data de cadastro do cliente (dd-mm-aaaa)", "Original (texto)"),
 "Recency": ("Dias desde a última compra", "Original"),
 "MntWines": ("Gasto com vinhos nos últimos 2 anos", "Original"), "MntFruits": ("Gasto com frutas nos últimos 2 anos", "Original"),
 "MntMeatProducts": ("Gasto com carnes nos últimos 2 anos", "Original"), "MntFishProducts": ("Gasto com peixes nos últimos 2 anos", "Original"),
 "MntSweetProducts": ("Gasto com doces nos últimos 2 anos", "Original"), "MntGoldProds": ("Gasto com produtos premium (ouro) nos últimos 2 anos", "Original"),
 "NumDealsPurchases": ("Compras feitas com desconto", "Original"), "NumWebPurchases": ("Compras pelo site", "Original"),
 "NumCatalogPurchases": ("Compras por catálogo", "Original"), "NumStorePurchases": ("Compras em loja física", "Original"),
 "NumWebVisitsMonth": ("Visitas ao site no último mês", "Original"),
 "AcceptedCmp1": ("Aceitou a oferta da campanha 1 (1=sim)", "Original"), "AcceptedCmp2": ("Aceitou a oferta da campanha 2 (1=sim)", "Original"),
 "AcceptedCmp3": ("Aceitou a oferta da campanha 3 (1=sim)", "Original"), "AcceptedCmp4": ("Aceitou a oferta da campanha 4 (1=sim)", "Original"),
 "AcceptedCmp5": ("Aceitou a oferta da campanha 5 (1=sim)", "Original"), "Response": ("Aceitou a oferta da última campanha (1=sim)", "Original"),
 "Complain": ("Reclamou nos últimos 2 anos (1=sim)", "Original"),
 "Z_CostContact": ("Constante (=3); significado não documentado; assumido como custo por contato (premissa P4)", "Original; descartada na Silver"),
 "Z_Revenue": ("Constante (=11); significado não documentado; assumido como receita por aceite (premissa P4)", "Original; descartada na Silver"),
})
# Colunas Silver/Gold
D.update({
 "cliente_id": ("Identificador sequencial do cliente (chave)", "Silver: renomeado de ID"),
 "ano_nascimento": ("Ano de nascimento (corrigido se implausível)", "Silver: Year_Birth; recalculado de idade quando corrigida (R5)"),
 "idade": ("Idade em anos na data de referência (maior data de cadastro)", "Silver: ano da data_ref − Year_Birth; >100 ou <16 → mediana (R2, R5)"),
 "renda": ("Renda anual do domicílio (imputada se nula/outlier)", "Silver: Income; nulos e outlier → mediana por educação (R3, R4)"),
 "n_filhos": ("Total de filhos no domicílio", "Silver: Kidhome + Teenhome"),
 "filhos_criancas": ("Nº de crianças no domicílio", "Silver: Kidhome"), "filhos_adolescentes": ("Nº de adolescentes no domicílio", "Silver: Teenhome"),
 "flag_renda_outlier": ("Renda original era outlier extremo", "Silver (R4)"), "flag_renda_imputada": ("Renda foi imputada (nula ou outlier)", "Silver (R3/R4)"),
 "flag_idade_corrigida": ("Idade foi corrigida por implausibilidade", "Silver (R5)"),
 "faixa_etaria": ("Faixa etária (até 34, 35-44, 45-54, 55-64, 65+)", "Gold: corte de idade"),
 "faixa_renda": ("Quartil de renda (Q1 menor … Q4 maior)", "Gold: qcut(renda, 4)"),
 "tem_filhos": ("Possui ao menos um filho no domicílio", "Gold: n_filhos > 0"),
 "educacao_id": ("Chave substituta da dimensão educação", "Gold: sequencial"),
 "educacao_original": ("Escolaridade como na origem", "Silver: Education"),
 "educacao": ("Escolaridade padronizada (Básico, Graduação, Mestrado, Doutorado)", "Silver: 2n Cycle e Master → Mestrado (R7)"),
 "educacao_nivel": ("Nível ordinal da escolaridade (1 a 4)", "Silver: mapeamento ordinal"),
 "estado_civil_id": ("Chave substituta da dimensão estado civil", "Gold: sequencial"),
 "estado_civil": ("Estado civil padronizado", "Silver: Alone→Solteiro(a); Absurd/YOLO→Não informado (R6)"),
 "estado_civil_grupo": ("Com parceiro(a) / Sem parceiro(a) / Não informado", "Silver (R6)"),
 "data_id": ("Chave da data de cadastro (aaaammdd)", "Gold: de Dt_Customer"), "data": ("Data de cadastro", "Silver: Dt_Customer convertido (R2)"),
 "ano": ("Ano da data", "Gold"), "trimestre": ("Trimestre da data", "Gold"), "mes": ("Mês da data", "Gold"),
 "dia_semana": ("Dia da semana (1=segunda)", "Gold"), "ano_mes": ("Ano-mês (aaaa-mm)", "Gold"),
 "data_cadastro": ("Data de cadastro do cliente", "Silver: Dt_Customer convertido (R2)"),
 "segmento_id": ("Segmento do cluster (1 = maior gasto médio)", "Gold: K-Means, ordenado por gasto médio"),
 "rotulo": ("Rótulo do segmento (S1 maior gasto … Sk menor gasto)", "Gold"),
 "n_clientes": ("Nº de clientes no segmento", "Gold: contagem"), "gasto_medio": ("Gasto total médio do segmento", "Gold: média"),
 "renda_media": ("Renda média do segmento", "Gold: média"), "recencia_media": ("Recência média (dias) do segmento", "Gold: média"),
 "recencia_dias": ("Dias desde a última compra", "Silver: Recency"),
 "tempo_cliente_dias": ("Dias entre o cadastro e a data de referência", "Silver: data_ref − Dt_Customer (R2, R8)"),
 "gasto_total": ("Gasto total nas 6 categorias", "Silver: soma de Mnt* (R8)"),
 "compras_total": ("Compras totais (web + catálogo + loja)", "Silver (R8); compras com desconto é subconjunto"),
 "compras_web": ("Compras pelo site", "Silver: NumWebPurchases"), "compras_catalogo": ("Compras por catálogo", "Silver: NumCatalogPurchases"),
 "compras_loja": ("Compras em loja física", "Silver: NumStorePurchases"), "compras_desconto": ("Compras com desconto", "Silver: NumDealsPurchases"),
 "visitas_web_mes": ("Visitas ao site no último mês", "Silver: NumWebVisitsMonth"),
 "ticket_medio": ("Gasto total ÷ compras totais (0 se sem compras)", "Silver (R8, R10)"),
 "share_web": ("% das compras pelo site", "Silver (R8)"), "share_catalogo": ("% das compras por catálogo", "Silver (R8)"),
 "share_loja": ("% das compras em loja", "Silver (R8)"),
 "pct_compras_desconto": ("Compras com desconto ÷ compras totais (máx. 1)", "Silver (R8, R9)"),
 "campanha_1": ("Aceitou campanha 1", "Silver: AcceptedCmp1"), "campanha_2": ("Aceitou campanha 2", "Silver: AcceptedCmp2"),
 "campanha_3": ("Aceitou campanha 3", "Silver: AcceptedCmp3"), "campanha_4": ("Aceitou campanha 4", "Silver: AcceptedCmp4"),
 "campanha_5": ("Aceitou campanha 5", "Silver: AcceptedCmp5"), "campanha_ultima": ("Aceitou a última campanha", "Silver: Response"),
 "n_campanhas_aceitas": ("Nº de campanhas aceitas entre 1 e 5", "Silver (R8)"),
 "reclamacao_2anos": ("Reclamou nos últimos 2 anos", "Silver: Complain"),
 "flag_sem_compras": ("Cliente sem nenhuma compra registrada", "Silver (R10)"),
})
for c in ["vinhos", "carnes", "peixes", "frutas", "doces", "ouro"]:
    D[f"gasto_{c}"] = (f"Gasto com {c}", "Silver: Mnt* correspondente")
    D[f"pct_gasto_{c}"] = (f"% do gasto total em {c}", "Silver (R8)")

LINHAGEM_TABELA = {
    "bronze": "Kaggle imakash3011/customer-personality-analysis → CSV bruto",
    "fato_cliente_vendas": "Bronze → Silver (silver_clientes) → Gold", "dim_cliente": "Bronze → Silver → Gold",
    "dim_educacao": "Bronze → Silver → Gold", "dim_estado_civil": "Bronze → Silver → Gold",
    "dim_tempo": "Bronze → Silver → Gold", "dim_segmento": "Silver → K-Means (Gold)",
}
EXTRACAO = meta["data_coleta_utc"][:10]

def tipo_dom(s, nome):
    if pd.api.types.is_bool_dtype(s): return "booleano", "{False, True}"
    if pd.api.types.is_datetime64_any_dtype(s): return "data", f"{s.min().date()} a {s.max().date()}"
    if pd.api.types.is_numeric_dtype(s):
        if nome.endswith("_id") or nome == "cliente_id" or nome == "ID": return "numérico (chave)", f"inteiro, {s.min()} a {s.max()}, únicos={s.nunique()}"
        return "numérico", f"{s.min():g} a {s.max():g}"
    vals = sorted(s.dropna().astype(str).unique())
    if len(vals) <= 12: return "categórico", "; ".join(vals)
    return "texto", f"{len(vals)} valores distintos"

def catalogar(tabela, df, linhagem_tab):
    linhas = []
    for c in df.columns:
        t, dom = tipo_dom(df[c], c)
        nn = int(df[c].isna().sum())
        obrig = "Obrigatório (0 nulos)" if nn == 0 else f"Admite nulos ({nn}); ausência = valor não informado na origem"
        desc, transf = D.get(c, ("[preencher]", "[preencher]"))
        linhas.append({"tabela": tabela, "atributo": c, "tipo": t, "descricao": desc, "dominio": dom,
                       "obrigatoriedade": obrig, "linhagem": f"{linhagem_tab}; extraído em {EXTRACAO}; {transf}"})
    return linhas

linhas = catalogar("bronze_marketing_campaign", raw, LINHAGEM_TABELA["bronze"])
for nome, df_ in gold.items():
    linhas += catalogar(nome, df_, LINHAGEM_TABELA[nome])
catalogo = pd.DataFrame(linhas)
faltando = catalogo[catalogo["descricao"] == "[preencher]"]["atributo"].unique().tolist()
if faltando: print("Atributos sem descrição:", faltando)
catalogo.to_csv(BASE / "catalogo" / "catalogo_dados.csv", index=False, encoding="utf-8")

md_cat = ["# Catálogo de Dados\n", f"Dataset: Kaggle `{KAGGLE_DATASET}` · Licença: {LICENCA_DADOS} · Extração: {EXTRACAO}\n"]
for tab, g in catalogo.groupby("tabela", sort=False):
    md_cat.append(f"\n## {tab}\n\n| Atributo | Tipo | Descrição | Domínio | Obrigatoriedade | Linhagem |\n|---|---|---|---|---|---|")
    for r in g.itertuples():
        md_cat.append(f"| {r.atributo} | {r.tipo} | {r.descricao} | {r.dominio} | {r.obrigatoriedade} | {r.linhagem} |")
(BASE / "catalogo" / "catalogo_dados.md").write_text("\n".join(md_cat), encoding="utf-8")
print(f"Catálogo: {len(catalogo)} atributos em {catalogo['tabela'].nunique()} tabelas")
catalogo.head(12)

## 9. Notebooks Databricks (Bronze → Silver → Gold → Análise)
As quatro células abaixo gravam os **modelos** dos notebooks (formato `# Databricks notebook source`). A célula seguinte preenche os parâmetros da seção 3, insere o núcleo `mvp_core.py` e salva os arquivos finais em `databricks/` e `notebooks/`.

| Notebook | Camada | O que faz |
|---|---|---|
| `01_bronze_ingestao` | Bronze | CSV bruto do Volume → Delta, sem alterações + metadados de ingestão |
| `02_silver_limpeza_features` | Silver | tipagem, nulos/outliers, padronização, features; grava log de regras |
| `03_gold_estrela_cluster` | Gold | K-Means (cotovelo, silhouette, estabilidade), esquema estrela, visões analíticas |
| `04_analise_p1_p5` | Análise | respostas P1–P5 em SQL/pandas com gráficos e discussão |

In [ ]:
%%writefile databricks/_templates/01_bronze_ingestao.py
# Databricks notebook source
# MAGIC %md
# MAGIC # 01 · Bronze — ingestão do CSV bruto
# MAGIC **Objetivo:** ler `marketing_campaign.csv` (separado por TAB) do Volume do Unity Catalog e gravar em Delta **sem nenhuma alteração de conteúdo**.
# MAGIC Todas as colunas entram como texto (sem inferência de tipos), mais metadados de ingestão (`_ingestao_ts`, `_arquivo_origem`, `_lote_id`).
# MAGIC Tipagem e limpeza ficam para a camada Silver.

# COMMAND ----------

# Configuração (preenchida pelo Colab; pode editar aqui se rodar manualmente)
CATALOG = "{{CATALOG}}"        # Free Edition: catálogo padrão costuma ser "workspace"
SCHEMA  = "{{SCHEMA}}"
VOLUME  = "{{VOLUME}}"
CSV_NAME = "{{CSV_NAME}}"
SEED = {{SEED}}
K_MAX = {{K_MAX}}
K_MIN_ESCOLHA = {{K_MIN_ESCOLHA}}
PCT_RISCO_GASTO = {{PCT_RISCO_GASTO}}   # cliente de alto gasto = acima deste percentil
PCT_RISCO_RECENCIA = {{PCT_RISCO_RECENCIA}}  # recência alta (muitos dias sem comprar) = acima deste percentil
N_CONTATOS_POR_CLIENTE = 6      # premissa P4: cada cliente recebeu as 6 campanhas (1 a 5 + última)

def T(nome):
    """Nome totalmente qualificado de uma tabela/view."""
    return f"{CATALOG}.{SCHEMA}.{nome}"

# COMMAND ----------

# 1. Garante schema e volume (idempotente)
import os, uuid
from pyspark.sql import functions as F

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")
caminho = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/{CSV_NAME}"
print("Arquivo esperado:", caminho)

# COMMAND ----------

# 2. Verifica se o CSV foi enviado ao Volume
if not os.path.exists(caminho):
    raise FileNotFoundError(
        f"CSV não encontrado em {caminho}. Envie o arquivo pela interface: "
        "Catalog > {CATALOG} > {SCHEMA} > {VOLUME} > Upload to this volume.".format(
            CATALOG=CATALOG, SCHEMA=SCHEMA, VOLUME=VOLUME))
print("Tamanho (bytes):", os.path.getsize(caminho))

# COMMAND ----------

# 3. Leitura bruta (tudo string) + metadados de ingestão
df = (spark.read.option("header", True).option("sep", "\t").option("inferSchema", False).csv(caminho))
df = (df.withColumn("_ingestao_ts", F.current_timestamp())
        .withColumn("_arquivo_origem", F.lit(caminho))
        .withColumn("_lote_id", F.lit(str(uuid.uuid4()))))

n = df.count()
print(f"Linhas: {n} | Colunas (com metadados): {len(df.columns)}")
if n == 0:
    raise ValueError("Bronze vazio: verifique o separador (TAB) e o arquivo.")

# COMMAND ----------

# 4. Grava Delta (overwrite para o job ser reexecutável)
(df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
   .saveAsTable(T("bronze_marketing_campaign")))
print("Tabela criada:", T("bronze_marketing_campaign"))
display(spark.sql(f"SELECT * FROM {T('bronze_marketing_campaign')} LIMIT 5"))

# COMMAND ----------

# 5. Evidência: histórico Delta e contagem
display(spark.sql(f"DESCRIBE HISTORY {T('bronze_marketing_campaign')}"))
display(spark.sql(f"SELECT COUNT(*) AS linhas FROM {T('bronze_marketing_campaign')}"))

In [ ]:
%%writefile databricks/_templates/02_silver_limpeza_features.py
# Databricks notebook source
# MAGIC %md
# MAGIC # 02 · Silver — tipagem, qualidade e features
# MAGIC Lê a Bronze, aplica as regras de negócio documentadas (R1–R10) e grava `silver_clientes`, mais duas tabelas de apoio:
# MAGIC `silver_regras_negocio` (cada regra e quantos registros afetou) e `silver_constantes` (valores das colunas constantes descartadas, usados como premissa em P4).
# MAGIC
# MAGIC | Regra | O que faz |
# MAGIC |---|---|
# MAGIC | R1 | Descarta colunas constantes (`Z_CostContact`, `Z_Revenue`) |
# MAGIC | R2 | Data de referência = maior `Dt_Customer` (não a data de hoje) |
# MAGIC | R3/R4 | Renda nula ou outlier extremo (> Q3 + 3·IQR) → imputada pela mediana do nível de educação |
# MAGIC | R5 | Idade > 100 ou < 16 → imputada pela mediana |
# MAGIC | R6/R7 | Padroniza estado civil e educação |
# MAGIC | R8–R10 | Features derivadas e checagens de consistência |

# COMMAND ----------

# Configuração (preenchida pelo Colab; pode editar aqui se rodar manualmente)
CATALOG = "{{CATALOG}}"        # Free Edition: catálogo padrão costuma ser "workspace"
SCHEMA  = "{{SCHEMA}}"
VOLUME  = "{{VOLUME}}"
CSV_NAME = "{{CSV_NAME}}"
SEED = {{SEED}}
K_MAX = {{K_MAX}}
K_MIN_ESCOLHA = {{K_MIN_ESCOLHA}}
PCT_RISCO_GASTO = {{PCT_RISCO_GASTO}}   # cliente de alto gasto = acima deste percentil
PCT_RISCO_RECENCIA = {{PCT_RISCO_RECENCIA}}  # recência alta (muitos dias sem comprar) = acima deste percentil
N_CONTATOS_POR_CLIENTE = 6      # premissa P4: cada cliente recebeu as 6 campanhas (1 a 5 + última)

def T(nome):
    """Nome totalmente qualificado de uma tabela/view."""
    return f"{CATALOG}.{SCHEMA}.{nome}"

# COMMAND ----------

# ---- Núcleo de lógica (mesmo código usado no Colab) ----
# {{CORE}}

# COMMAND ----------

# 1. Lê a Bronze (volume pequeno: converter para pandas é seguro e simplifica as regras)
bronze = spark.table(T("bronze_marketing_campaign")).toPandas()
print("Bronze:", bronze.shape)

# COMMAND ----------

# 2. Aplica as regras de negócio (lógica em silver_transform)
silver, log_regras, constantes = silver_transform(bronze)
print("Silver:", silver.shape)
display(log_regras)

# COMMAND ----------

# 3. Checagens automáticas antes de gravar
assert len(silver) == len(bronze), "Silver perdeu/duplicou linhas"
assert silver["cliente_id"].is_unique, "cliente_id duplicado"
assert silver.isna().sum().sum() == 0, "Silver não deveria ter nulos após o tratamento"
assert (silver["gasto_total"] >= 0).all()
print("Checagens OK")

# COMMAND ----------

# 4. Grava Delta
import pandas as pd
(spark.createDataFrame(silver).write.format("delta").mode("overwrite")
      .option("overwriteSchema", "true").saveAsTable(T("silver_clientes")))
(spark.createDataFrame(log_regras.astype(str)).write.format("delta").mode("overwrite")
      .option("overwriteSchema", "true").saveAsTable(T("silver_regras_negocio")))
const_df = pd.DataFrame({"coluna": list(constantes.keys()), "valor": list(constantes.values())})
(spark.createDataFrame(const_df).write.format("delta").mode("overwrite")
      .option("overwriteSchema", "true").saveAsTable(T("silver_constantes")))
display(spark.sql(f"SELECT COUNT(*) AS linhas FROM {T('silver_clientes')}"))

In [ ]:
%%writefile databricks/_templates/03_gold_estrela_cluster.py
# Databricks notebook source
# MAGIC %md
# MAGIC # 03 · Gold — esquema estrela, clusterização e visões analíticas
# MAGIC 1. Clusteriza os clientes com **K-Means** sobre RFM + perfil (`recencia_dias`, `compras_total`, `log(gasto_total)`, `renda`, `idade`, `n_filhos`), padronizados (z-score).
# MAGIC 2. Escolhe **k** pelo maior *silhouette* entre `K_MIN_ESCOLHA` e `K_MAX` (cotovelo/inércia é registrado como apoio) e mede a **estabilidade** com 5 sementes (ARI).
# MAGIC 3. Grava o esquema estrela: `fato_cliente_vendas`, `dim_cliente`, `dim_educacao`, `dim_estado_civil`, `dim_tempo`, `dim_segmento`.
# MAGIC 4. Cria as visões analíticas usadas em `04_analise`.

# COMMAND ----------

# Configuração (preenchida pelo Colab; pode editar aqui se rodar manualmente)
CATALOG = "{{CATALOG}}"        # Free Edition: catálogo padrão costuma ser "workspace"
SCHEMA  = "{{SCHEMA}}"
VOLUME  = "{{VOLUME}}"
CSV_NAME = "{{CSV_NAME}}"
SEED = {{SEED}}
K_MAX = {{K_MAX}}
K_MIN_ESCOLHA = {{K_MIN_ESCOLHA}}
PCT_RISCO_GASTO = {{PCT_RISCO_GASTO}}   # cliente de alto gasto = acima deste percentil
PCT_RISCO_RECENCIA = {{PCT_RISCO_RECENCIA}}  # recência alta (muitos dias sem comprar) = acima deste percentil
N_CONTATOS_POR_CLIENTE = 6      # premissa P4: cada cliente recebeu as 6 campanhas (1 a 5 + última)

def T(nome):
    """Nome totalmente qualificado de uma tabela/view."""
    return f"{CATALOG}.{SCHEMA}.{nome}"

# COMMAND ----------

# ---- Núcleo de lógica (mesmo código usado no Colab) ----
# {{CORE}}

# COMMAND ----------

# 1. Lê Silver e clusteriza
silver = spark.table(T("silver_clientes")).toPandas()
seg_id, metricas, estab = clusterizar(silver, k_max=K_MAX, k_min_escolha=K_MIN_ESCOLHA, seed=SEED)
print("k escolhido:", estab["k"], "| silhouette:", round(estab["silhouette_k"], 3))
print("Estabilidade (ARI entre sementes): média", round(estab["ari_medio"], 3), "| mínimo", round(estab["ari_min"], 3))
display(metricas)

# COMMAND ----------

# 2. Constrói e grava o esquema estrela
import pandas as pd
gold = construir_gold(silver, seg_id)
for nome, pdf in gold.items():
    (spark.createDataFrame(pdf).write.format("delta").mode("overwrite")
          .option("overwriteSchema", "true").saveAsTable(T(nome)))
    print(f"{nome}: {len(pdf)} linhas")

# métricas da clusterização e estabilidade (apoiam a justificativa de k)
(spark.createDataFrame(metricas).write.format("delta").mode("overwrite")
      .option("overwriteSchema", "true").saveAsTable(T("gold_clusterizacao_metricas")))
estab_df = pd.DataFrame([{"k": estab["k"], "silhouette": estab["silhouette_k"], "ari_medio": estab["ari_medio"],
                          "ari_min": estab["ari_min"], "sementes": str(estab["sementes"]), "seed_base": SEED}])
(spark.createDataFrame(estab_df).write.format("delta").mode("overwrite")
      .option("overwriteSchema", "true").saveAsTable(T("gold_clusterizacao_estabilidade")))

# premissas de P4 (Z_* são premissa: a base não documenta o significado)
const = {r["coluna"]: r["valor"] for r in spark.table(T("silver_constantes")).collect()}
prem = pd.DataFrame([{"custo_contato": const["Z_CostContact"], "receita_aceite": const["Z_Revenue"],
                      "contatos_por_cliente": N_CONTATOS_POR_CLIENTE}])
(spark.createDataFrame(prem).write.format("delta").mode("overwrite")
      .option("overwriteSchema", "true").saveAsTable(T("gold_premissas_p4")))
display(prem)

# COMMAND ----------

# 3. Visões analíticas por segmento (SQL)
spark.sql(f"""
CREATE OR REPLACE VIEW {T('vw_perfil_segmento')} AS
SELECT s.segmento_id, s.rotulo,
       COUNT(*) AS n_clientes,
       ROUND(AVG(c.renda),0) AS renda_media,
       ROUND(AVG(c.idade),1) AS idade_media,
       ROUND(AVG(c.n_filhos),2) AS filhos_medio,
       ROUND(AVG(CASE WHEN c.tem_filhos THEN 1 ELSE 0 END),3) AS pct_com_filhos,
       ROUND(AVG(f.gasto_total),0) AS gasto_medio,
       ROUND(AVG(f.compras_total),1) AS compras_media,
       ROUND(AVG(f.recencia_dias),1) AS recencia_media,
       ROUND(AVG(f.ticket_medio),1) AS ticket_medio,
       ROUND(AVG(f.visitas_web_mes),2) AS visitas_web_media,
       ROUND(AVG(f.tempo_cliente_dias),0) AS tempo_cliente_medio
FROM {T('fato_cliente_vendas')} f
JOIN {T('dim_cliente')} c ON f.cliente_id = c.cliente_id
JOIN {T('dim_segmento')} s ON f.segmento_id = s.segmento_id
GROUP BY s.segmento_id, s.rotulo""")

spark.sql(f"""
CREATE OR REPLACE VIEW {T('vw_receita_segmento')} AS
SELECT s.segmento_id, s.rotulo,
       COUNT(*) AS n_clientes,
       SUM(f.gasto_total) AS gasto_total_segmento,
       ROUND(SUM(f.gasto_total) / SUM(SUM(f.gasto_total)) OVER (), 4) AS pct_receita,
       ROUND(COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct_clientes,
       ROUND(AVG(f.gasto_total),1) AS gasto_medio_cliente
FROM {T('fato_cliente_vendas')} f
JOIN {T('dim_segmento')} s ON f.segmento_id = s.segmento_id
GROUP BY s.segmento_id, s.rotulo""")

spark.sql(f"""
CREATE OR REPLACE VIEW {T('vw_mix_canal_segmento')} AS
SELECT s.segmento_id, s.rotulo,
       SUM(f.gasto_vinhos)/SUM(f.gasto_total) AS mix_vinhos,
       SUM(f.gasto_carnes)/SUM(f.gasto_total) AS mix_carnes,
       SUM(f.gasto_peixes)/SUM(f.gasto_total) AS mix_peixes,
       SUM(f.gasto_frutas)/SUM(f.gasto_total) AS mix_frutas,
       SUM(f.gasto_doces)/SUM(f.gasto_total) AS mix_doces,
       SUM(f.gasto_ouro)/SUM(f.gasto_total) AS mix_ouro,
       SUM(f.compras_loja)/SUM(f.compras_total) AS canal_loja,
       SUM(f.compras_web)/SUM(f.compras_total) AS canal_web,
       SUM(f.compras_catalogo)/SUM(f.compras_total) AS canal_catalogo,
       SUM(f.compras_desconto)/SUM(f.compras_total) AS pct_compras_desconto,
       AVG(f.visitas_web_mes) AS visitas_web_media
FROM {T('fato_cliente_vendas')} f
JOIN {T('dim_segmento')} s ON f.segmento_id = s.segmento_id
GROUP BY s.segmento_id, s.rotulo""")

spark.sql(f"""
CREATE OR REPLACE VIEW {T('vw_campanhas_segmento')} AS
SELECT s.segmento_id, s.rotulo, COUNT(*) AS n_clientes,
       AVG(f.campanha_1) AS taxa_c1, AVG(f.campanha_2) AS taxa_c2, AVG(f.campanha_3) AS taxa_c3,
       AVG(f.campanha_4) AS taxa_c4, AVG(f.campanha_5) AS taxa_c5, AVG(f.campanha_ultima) AS taxa_ultima,
       SUM(f.campanha_1+f.campanha_2+f.campanha_3+f.campanha_4+f.campanha_5+f.campanha_ultima) AS aceites,
       COUNT(*) * MAX(p.contatos_por_cliente) AS contatos,
       COUNT(*) * MAX(p.contatos_por_cliente) * MAX(p.custo_contato) AS custo_estimado,
       SUM(f.campanha_1+f.campanha_2+f.campanha_3+f.campanha_4+f.campanha_5+f.campanha_ultima) * MAX(p.receita_aceite) AS receita_estimada
FROM {T('fato_cliente_vendas')} f
JOIN {T('dim_segmento')} s ON f.segmento_id = s.segmento_id
CROSS JOIN {T('gold_premissas_p4')} p
GROUP BY s.segmento_id, s.rotulo""")

spark.sql(f"""
CREATE OR REPLACE VIEW {T('vw_clientes_risco')} AS
WITH lim AS (
  SELECT PERCENTILE_APPROX(gasto_total, {PCT_RISCO_GASTO}) AS lim_gasto,
         PERCENTILE_APPROX(recencia_dias, {PCT_RISCO_RECENCIA}) AS lim_rec
  FROM {T('fato_cliente_vendas')})
SELECT f.cliente_id, s.rotulo, f.gasto_total, f.recencia_dias, f.compras_total,
       f.compras_loja, f.compras_web, f.compras_catalogo, f.n_campanhas_aceitas, f.campanha_ultima
FROM {T('fato_cliente_vendas')} f
JOIN {T('dim_segmento')} s ON f.segmento_id = s.segmento_id
CROSS JOIN lim
WHERE f.gasto_total >= lim.lim_gasto AND f.recencia_dias >= lim.lim_rec""")
print("Visões criadas")
display(spark.sql(f"SELECT * FROM {T('vw_perfil_segmento')} ORDER BY segmento_id"))

In [ ]:
%%writefile databricks/_templates/04_analise_p1_p5.py
# Databricks notebook source
# MAGIC %md
# MAGIC # 04 · Análise — respostas a P1–P5
# MAGIC Cada seção traz: consulta (SQL/pandas) → gráfico → **discussão**. Todos os números vêm da execução; os trechos entre colchetes `[preencher ...]` devem ser completados **depois** de rodar, olhando as saídas.
# MAGIC Resultados fracos ou inconclusivos devem ser relatados como tal.

# COMMAND ----------

# Configuração (preenchida pelo Colab; pode editar aqui se rodar manualmente)
CATALOG = "{{CATALOG}}"        # Free Edition: catálogo padrão costuma ser "workspace"
SCHEMA  = "{{SCHEMA}}"
VOLUME  = "{{VOLUME}}"
CSV_NAME = "{{CSV_NAME}}"
SEED = {{SEED}}
K_MAX = {{K_MAX}}
K_MIN_ESCOLHA = {{K_MIN_ESCOLHA}}
PCT_RISCO_GASTO = {{PCT_RISCO_GASTO}}   # cliente de alto gasto = acima deste percentil
PCT_RISCO_RECENCIA = {{PCT_RISCO_RECENCIA}}  # recência alta (muitos dias sem comprar) = acima deste percentil
N_CONTATOS_POR_CLIENTE = 6      # premissa P4: cada cliente recebeu as 6 campanhas (1 a 5 + última)

def T(nome):
    """Nome totalmente qualificado de uma tabela/view."""
    return f"{CATALOG}.{SCHEMA}.{nome}"

# COMMAND ----------

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "font.size": 10})
def q(sql):
    return spark.sql(sql).toPandas()
def seg_cores(rotulos):
    paleta = ["#2a6f97", "#e07a1f", "#6a994e", "#b5446e", "#7a6fb5", "#8d8d8d", "#c9a227", "#2a9d8f"]
    return {r: paleta[i % len(paleta)] for i, r in enumerate(sorted(rotulos))}
perfil = q(f"SELECT * FROM {T('vw_perfil_segmento')} ORDER BY segmento_id")
CORES = seg_cores(perfil["rotulo"])

# COMMAND ----------

# MAGIC %md
# MAGIC ## P1 · Quantos segmentos naturais existem e qual o perfil de cada um?

# COMMAND ----------

# P1a. Escolha de k: cotovelo (inércia) e silhouette
m = q(f"SELECT * FROM {T('gold_clusterizacao_metricas')} ORDER BY k")
est = q(f"SELECT * FROM {T('gold_clusterizacao_estabilidade')}")
k_esc = int(est["k"].iloc[0])
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].plot(m["k"], m["inercia"], marker="o", color="#2a6f97"); ax[0].set_title("Método do cotovelo (inércia)")
ax[0].set_xlabel("k"); ax[0].axvline(k_esc, ls="--", color="#999")
ax[1].plot(m["k"], m["silhouette"], marker="o", color="#e07a1f"); ax[1].set_title("Silhouette médio")
ax[1].set_xlabel("k"); ax[1].axvline(k_esc, ls="--", color="#999")
plt.tight_layout(); plt.show()
display(m); display(est)

# COMMAND ----------

# P1b. Perfil dos segmentos (tabela + mapa de calor padronizado por coluna)
display(perfil)
cols = ["renda_media", "idade_media", "filhos_medio", "gasto_medio", "compras_media", "recencia_media", "ticket_medio", "visitas_web_media"]
z = (perfil[cols] - perfil[cols].mean()) / perfil[cols].std(ddof=0).replace(0, 1)
fig, ax = plt.subplots(figsize=(9, 0.9 + 0.7 * len(perfil)))
im = ax.imshow(z.values, cmap="RdBu_r", vmin=-1.6, vmax=1.6, aspect="auto")
ax.set_xticks(range(len(cols))); ax.set_xticklabels(cols, rotation=30, ha="right")
ax.set_yticks(range(len(perfil))); ax.set_yticklabels(perfil["rotulo"])
for i in range(len(perfil)):
    for j, c in enumerate(cols):
        ax.text(j, i, f"{perfil[c].iloc[i]:,.1f}", ha="center", va="center", fontsize=8)
ax.set_title("Perfil por segmento (cor = desvio em relação à média dos segmentos)")
plt.colorbar(im, ax=ax, shrink=0.8); plt.tight_layout(); plt.show()

# COMMAND ----------

# MAGIC %md
# MAGIC ### Discussão P1
# MAGIC - **Número de segmentos:** o silhouette indicou k = **[preencher após executar]** com valor **[preencher]**. Valores de silhouette abaixo de ~0,35 indicam grupos **pouco separados**: se for o caso, relate que os segmentos são mais um *contínuo de valor* (renda/gasto) do que grupos naturais bem delimitados.
# MAGIC - **Por que não k=2:** [preencher: k=2 costuma separar apenas "alto" × "baixo" valor, o que é pouco acionável; a escolha de k ≥ `K_MIN_ESCOLHA` é uma decisão de negócio, declarada].
# MAGIC - **Estabilidade:** ARI médio entre sementes = **[preencher]** (próximo de 1 indica partição reprodutível).
# MAGIC - **Perfis:** [preencher: descreva cada segmento em renda, filhos, gasto e frequência olhando a tabela/mapa de calor].

# COMMAND ----------

# MAGIC %md
# MAGIC ## P2 · Contribuição de cada segmento para a receita e concentração (Pareto 80%)

# COMMAND ----------

rec = q(f"SELECT * FROM {T('vw_receita_segmento')} ORDER BY segmento_id")
display(rec)
x = np.arange(len(rec)); w = 0.38
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.bar(x - w/2, rec["pct_clientes"] * 100, w, label="% dos clientes", color="#bdbdbd")
ax.bar(x + w/2, rec["pct_receita"] * 100, w, label="% do gasto (receita)", color="#2a6f97")
ax.set_xticks(x); ax.set_xticklabels(rec["rotulo"], rotation=15); ax.set_ylabel("%"); ax.legend()
ax.set_title("Participação no gasto total × participação nos clientes")
plt.tight_layout(); plt.show()

# Pareto: fração de clientes que concentra 80% do gasto
g = q(f"SELECT cliente_id, gasto_total FROM {T('fato_cliente_vendas')}").sort_values("gasto_total", ascending=False).reset_index(drop=True)
g["pct_acum_gasto"] = g["gasto_total"].cumsum() / g["gasto_total"].sum()
g["pct_acum_clientes"] = (g.index + 1) / len(g)
n80 = int((g["pct_acum_gasto"] < 0.80).sum() + 1)
frac80 = n80 / len(g)
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(g["pct_acum_clientes"] * 100, g["pct_acum_gasto"] * 100, color="#2a6f97")
ax.axhline(80, ls="--", color="#999"); ax.axvline(frac80 * 100, ls="--", color="#999")
ax.set_xlabel("% acumulado de clientes (do maior para o menor gasto)"); ax.set_ylabel("% acumulado do gasto")
ax.set_title(f"Pareto: {frac80:.1%} dos clientes ({n80}) concentram 80% do gasto")
plt.tight_layout(); plt.show()
print(f"{n80} clientes ({frac80:.1%}) concentram 80% do gasto total.")

# COMMAND ----------

# MAGIC %md
# MAGIC ### Discussão P2
# MAGIC - **Receita por segmento:** o segmento **[preencher]** responde por **[preencher]%** do gasto com **[preencher]%** dos clientes; gasto médio por cliente: [preencher].
# MAGIC - **Pareto:** **[preencher]%** dos clientes concentram 80% do gasto. Se for bem acima de 20%, a concentração é **menor que a regra 80/20 clássica** — diga isso explicitamente.
# MAGIC - **Implicação comercial:** [preencher: quanto vale priorizar o segmento de maior gasto].

# COMMAND ----------

# MAGIC %md
# MAGIC ## P3 · Mix de categorias, canais, visitas web e compras com desconto

# COMMAND ----------

mix = q(f"SELECT * FROM {T('vw_mix_canal_segmento')} ORDER BY segmento_id")
display(mix)
cats = ["mix_vinhos", "mix_carnes", "mix_peixes", "mix_frutas", "mix_doces", "mix_ouro"]
canais = ["canal_loja", "canal_web", "canal_catalogo"]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for a, cols, titulo in [(ax[0], cats, "Mix de categorias (% do gasto)"), (ax[1], canais, "Canal (% das compras)")]:
    base = np.zeros(len(mix))
    for c in cols:
        a.bar(mix["rotulo"], mix[c] * 100, bottom=base, label=c.split("_")[1]); base += mix[c].values * 100
    a.set_title(titulo); a.set_ylabel("%"); a.tick_params(axis="x", rotation=15); a.legend(fontsize=8, ncol=2)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(9, 3.4))
ax[0].bar(mix["rotulo"], mix["visitas_web_media"], color="#7a6fb5"); ax[0].set_title("Visitas web/mês (média)")
ax[1].bar(mix["rotulo"], mix["pct_compras_desconto"] * 100, color="#b5446e"); ax[1].set_title("% das compras com desconto")
for a in ax: a.tick_params(axis="x", rotation=15)
plt.tight_layout(); plt.show()

# COMMAND ----------

# MAGIC %md
# MAGIC ### Discussão P3
# MAGIC - **Mix:** vinhos representam [preencher]% do gasto no total; os segmentos diferem principalmente em [preencher: categorias com maior diferença].
# MAGIC - **Canal:** a loja física concentra [preencher]% das compras; [preencher: segmento que mais usa web e catálogo].
# MAGIC - **Visitas web e desconto:** [preencher: relação entre muitas visitas web, compras com desconto e baixo gasto — **descreva como associação**, não como causalidade; a base não permite inferir causa].

# COMMAND ----------

# MAGIC %md
# MAGIC ## P4 · Aceitação de campanhas e retorno estimado por segmento
# MAGIC **Premissa declarada:** a base **não documenta** `Z_CostContact` e `Z_Revenue`. Assumimos custo por contato = `Z_CostContact` (3) e receita por aceite = `Z_Revenue` (11), e que **cada cliente foi contatado nas 6 campanhas** (1 a 5 e a última). Os resultados abaixo dependem inteiramente dessa premissa.

# COMMAND ----------

camp = q(f"SELECT * FROM {T('vw_campanhas_segmento')} ORDER BY segmento_id")
camp["retorno_estimado"] = camp["receita_estimada"] - camp["custo_estimado"]
camp["roi_estimado"] = camp["retorno_estimado"] / camp["custo_estimado"]
camp["retorno_por_contato"] = camp["retorno_estimado"] / camp["contatos"]
display(camp)
taxas = ["taxa_c1", "taxa_c2", "taxa_c3", "taxa_c4", "taxa_c5", "taxa_ultima"]
fig, ax = plt.subplots(figsize=(8, 0.9 + 0.7 * len(camp)))
im = ax.imshow(camp[taxas].values * 100, cmap="Blues", aspect="auto")
ax.set_xticks(range(6)); ax.set_xticklabels(["Cmp1", "Cmp2", "Cmp3", "Cmp4", "Cmp5", "Última"])
ax.set_yticks(range(len(camp))); ax.set_yticklabels(camp["rotulo"])
for i in range(len(camp)):
    for j, c in enumerate(taxas):
        ax.text(j, i, f"{camp[c].iloc[i]*100:.1f}%", ha="center", va="center", fontsize=8)
ax.set_title("Taxa de aceitação por campanha e segmento"); plt.tight_layout(); plt.show()
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.bar(camp["rotulo"], camp["roi_estimado"], color=[CORES[r] for r in camp["rotulo"]])
ax.axhline(0, color="#444", lw=0.8); ax.set_ylabel("ROI estimado"); ax.set_title("ROI estimado por segmento (sob a premissa de Z_*)")
ax.tick_params(axis="x", rotation=15); plt.tight_layout(); plt.show()

# COMMAND ----------

# MAGIC %md
# MAGIC ### Discussão P4
# MAGIC - **Aceitação:** a taxa varia de [preencher] a [preencher] entre segmentos; [preencher: campanha com menor/maior aceitação no geral — a base mostra taxas baixas (< ~15%), o que limita a precisão das comparações em segmentos pequenos].
# MAGIC - **Retorno estimado:** o segmento com melhor retorno por contato é **[preencher]** (ROI [preencher]). **Lembrete:** é estimativa sob premissa; se Z_* tiver outro significado, a conclusão pode mudar.
# MAGIC - **Limite:** `Response` refere-se a **uma única campanha**; não há dados de custo real nem de margem.

# COMMAND ----------

# MAGIC %md
# MAGIC ## P5 · Clientes de alto gasto e recência alta (risco de inatividade)
# MAGIC Definição: gasto total ≥ percentil `PCT_RISCO_GASTO` **e** dias desde a última compra ≥ percentil `PCT_RISCO_RECENCIA`. (Na base, `Recency` = dias desde a última compra; valor alto = cliente parado há mais tempo.)

# COMMAND ----------

risco = q(f"SELECT * FROM {T('vw_clientes_risco')} ORDER BY gasto_total DESC")
print(f"{len(risco)} clientes em risco ({len(risco)/spark.table(T('fato_cliente_vendas')).count():.1%} da base) "
      f"| gasto somado: {risco['gasto_total'].sum():,.0f}")
display(risco.head(20))
por_seg = risco.groupby("rotulo").agg(clientes=("cliente_id", "count"), gasto=("gasto_total", "sum"), recencia_media=("recencia_dias", "mean")).reset_index()
display(por_seg)
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.bar(por_seg["rotulo"], por_seg["clientes"], color=[CORES[r] for r in por_seg["rotulo"]])
ax.set_title("Clientes em risco por segmento"); ax.tick_params(axis="x", rotation=15)
plt.tight_layout(); plt.show()

# COMMAND ----------

# MAGIC %md
# MAGIC ### Discussão P5
# MAGIC - **Tamanho do risco:** [preencher] clientes (gasto somado [preencher]) estão em alto gasto + recência alta, concentrados em [preencher: segmento].
# MAGIC - **Ação sugerida:** contato proativo e personalizado pelo canal preferido do cliente (loja/web/catálogo), com oferta ligada à categoria dominante dele (ver P3); medir reativação em 30–60 dias.
# MAGIC - **Cautela:** recência alta pode refletir ciclo de compra natural (ex.: vinho premium), não abandono; validar com histórico de compras, que a base não traz.

# COMMAND ----------

# MAGIC %md
# MAGIC ## Discussão geral e playbook comercial
# MAGIC **Hipótese testada:** existem segmentos com valor, comportamento e resposta a campanhas diferentes? **[preencher: confirmada / parcialmente confirmada / não confirmada, com base em P1–P4 e na separação dos clusters]**.
# MAGIC
# MAGIC ### Playbook (uma ação por segmento) — complete após ver os perfis
# MAGIC | Segmento | Perfil resumido | Oferta | Canal | Prioridade |
# MAGIC |---|---|---|---|---|
# MAGIC | S1 - maior gasto | [preencher] | [ex.: curadoria premium/fidelização, conforme categoria dominante] | [preencher conforme canal dominante] | [alta, se ROI e receita altos] |
# MAGIC | S2 e demais intermediários | [preencher] | [ex.: cross-sell para a 2ª categoria mais comprada] | [preencher] | [preencher] |
# MAGIC | S_último - menor gasto | [preencher] | [ex.: ofertas de entrada/desconto, custo baixo] | [ex.: web/e-mail, de menor custo] | [baixa/automatizada] |
# MAGIC | Clientes em risco (P5) | alto gasto + parados | reativação personalizada | canal preferido | alta |
# MAGIC
# MAGIC ### Limitações
# MAGIC Dados de ~2012–2014 (podem estar defasados); sem custo/margem por produto; significado de Z_* é premissa; `Response` é de uma campanha; amostra com possível viés de seleção; segmentos pouco separados limitam a generalização.

In [ ]:
# Gera os notebooks finais: preenche parâmetros + insere o núcleo de lógica
import ast
core_texto = (BASE / "scripts" / "mvp_core.py").read_text()
params = {"CATALOG": CATALOG, "SCHEMA": SCHEMA, "VOLUME": VOLUME, "CSV_NAME": CSV_NAME, "SEED": SEED,
          "K_MAX": K_MAX, "K_MIN_ESCOLHA": K_MIN_ESCOLHA, "PCT_RISCO_GASTO": PCT_RISCO_GASTO,
          "PCT_RISCO_RECENCIA": PCT_RISCO_RECENCIA}
gerados = []
for tpl in sorted((BASE / "databricks" / "_templates").glob("*.py")):
    t = tpl.read_text()
    t = t.replace("# {{CORE}}", core_texto)
    for k, v in params.items():
        t = t.replace("{{" + k + "}}", str(v))
    sobra = [p for p in ["{{CATALOG}}", "{{SCHEMA}}", "{{VOLUME}}", "{{SEED}}"] if p in t]
    if sobra:
        raise ValueError(f"Placeholders não preenchidos em {tpl.name}: {sobra}")
    codigo = "\n".join(l for l in t.splitlines() if not l.startswith("# MAGIC"))
    ast.parse(codigo)                      # valida a sintaxe antes de entregar
    for pasta in ["databricks", "notebooks"]:
        (BASE / pasta / tpl.name).write_text(t)
    gerados.append(tpl.name)
print("Notebooks Databricks gerados e validados:", *gerados, sep="\n  - ")

## 10. Levar os dados e notebooks ao Databricks (Free Edition)

### Plano A — manual pela interface (recomendado, mais confiável)
1. Crie sua conta em **Databricks Free Edition** e abra o workspace.
2. **Notebooks:** *Workspace → Import* e envie os 4 arquivos `.py` de `databricks/` (formato *source* é reconhecido automaticamente).
3. **CSV:** em *Catalog* abra `workspace` (ou seu catálogo) → crie o schema `mvp_marketing` e o volume `raw` (o notebook 01 também cria, se não existirem) → **Upload to this volume** → envie `marketing_campaign.csv`.
4. Execute os notebooks **01 → 02 → 03 → 04** (computação serverless). Confira se `CATALOG` no topo bate com o seu catálogo.
5. **Job (valoriza o ETL):** *Jobs & Pipelines → Create job*, com 4 tarefas encadeadas (Bronze → Silver → Gold → Análise) apontando para os notebooks. O arquivo `databricks/job_definition.json` gerado abaixo traz a definição.

### Plano B — se algo falhar no Free Edition
Limitações comuns: não há clusters próprios (só serverless), cotas diárias de computação, e criação de catálogos/DBFS público restrita. Se o upload ao Volume falhar, grave em *Catalog → Create table → Upload file* e ajuste a leitura do notebook 01; se o job não puder ser criado, rode os notebooks em sequência e mostre o histórico de execução como evidência.

### Plano C — automatizado via Databricks SDK (opcional)
Defina `USAR_SDK = True` na seção 3. Usa token pessoal (Settings → Developer → Access tokens) lido por `getpass` — **nunca** escreva o token no notebook.

In [ ]:
# Definição do job (Bronze > Silver > Gold > Análise) para criação manual ou via SDK
ordem_tarefas = ["01_bronze_ingestao", "02_silver_limpeza_features", "03_gold_estrela_cluster", "04_analise_p1_p5"]
def definicao_job(pasta_workspace):
    tarefas = []
    for i, n in enumerate(ordem_tarefas):
        t = {"task_key": n, "notebook_task": {"notebook_path": f"{pasta_workspace}/{n}"}}
        if i: t["depends_on"] = [{"task_key": ordem_tarefas[i - 1]}]
        tarefas.append(t)
    return {"name": "MVP_UnB_Pipeline_Medalhao", "tasks": tarefas, "queue": {"enabled": True}}
job_json = definicao_job("/Workspace/Users/<seu-usuario>/mvp_unb")   # ajuste o usuário se criar manualmente
(BASE / "databricks" / "job_definition.json").write_text(json.dumps(job_json, indent=2))
print(json.dumps(job_json, indent=2)[:700], "...")

In [ ]:
# Plano C (opcional): envio automático via Databricks SDK
if not USAR_SDK:
    print("USAR_SDK = False -> use o Plano A (manual). Nada foi enviado.")
else:
    from getpass import getpass
    import base64
    try:
        from databricks.sdk import WorkspaceClient
        from databricks.sdk.service.workspace import ImportFormat, Language
        from databricks.sdk.service import jobs as dj
        host = input("URL do workspace (ex.: https://dbc-xxxx.cloud.databricks.com): ").strip()
        token = getpass("Token pessoal do Databricks (não será exibido): ")
        w = WorkspaceClient(host=host, token=token)
        usuario = w.current_user.me().user_name
        print("Autenticado como:", usuario)

        # schema e volume
        for criar in (lambda: w.schemas.create(name=SCHEMA, catalog_name=CATALOG),
                      lambda: w.volumes.create(catalog_name=CATALOG, schema_name=SCHEMA, name=VOLUME,
                                               volume_type=__import__("databricks.sdk.service.catalog", fromlist=["VolumeType"]).VolumeType.MANAGED)):
            try: criar()
            except Exception as e: print("  (ok/ignorado):", str(e)[:120])

        # CSV -> Volume
        destino_vol = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/{CSV_NAME}"
        with open(BASE / "dados" / CSV_NAME, "rb") as f:
            w.files.upload(destino_vol, f, overwrite=True)
        print("CSV enviado para", destino_vol)

        # notebooks -> workspace
        pasta_ws = f"/Workspace/Users/{usuario}/mvp_unb"
        w.workspace.mkdirs(pasta_ws)
        for n in ordem_tarefas:
            conteudo = (BASE / "databricks" / f"{n}.py").read_bytes()
            w.workspace.import_(path=f"{pasta_ws}/{n}", content=base64.b64encode(conteudo).decode(),
                                format=ImportFormat.SOURCE, language=Language.PYTHON, overwrite=True)
        print("Notebooks importados em", pasta_ws)

        # job
        tarefas = [dj.Task(task_key=t["task_key"], notebook_task=dj.NotebookTask(notebook_path=t["notebook_task"]["notebook_path"]),
                           depends_on=[dj.TaskDependency(task_key=d["task_key"]) for d in t.get("depends_on", [])])
                   for t in definicao_job(pasta_ws)["tasks"]]
        job = w.jobs.create(name="MVP_UnB_Pipeline_Medalhao", tasks=tarefas)
        print("Job criado. ID:", job.job_id, "-> execute em Jobs & Pipelines (ou w.jobs.run_now).")
    except Exception as e:
        print("Falha no Plano C:", repr(e)[:400])
        print("Use o Plano A (manual). O Free Edition restringe parte da API; isso é esperado.")

## 11. Ética, licenciamento e LGPD (seção 4 do PDF)
- **Licença:** a base declara **CC0 (domínio público)**; permite redistribuição e uso derivado. *Confirme na página do dataset e registre a data (seção 3).* Mesmo assim, o repositório referencia a fonte (link do Kaggle) e o `sha256`; o CSV bruto pode ser versionado ou substituído pelo script de coleta.
- **Dados pessoais (Lei 13.709/2018):** o conjunto **não tem identificadores diretos** (nome, CPF, e-mail, endereço). `ID` é sequencial, sem ligação com sistema real. Há **quase-identificadores** (ano de nascimento, escolaridade, estado civil, renda, filhos).
- **Risco de reidentificação:** medido abaixo por *k*-anonimato sobre combinações de quase-identificadores. Procedimento adotado: usar **faixas** (faixa etária, quartil de renda) nas dimensões analíticas e nunca publicar listas individuais com o conjunto completo de atributos; as listagens de clientes (P5) ficam só no Databricks e o repositório mostra no máximo `cliente_id` + métricas agregadas.
- **Dados corporativos:** não se aplica (base pública).

In [ ]:
# k-anonimato: tamanho do menor grupo de clientes que compartilham o mesmo conjunto de quase-identificadores
def k_anon(df, cols):
    tam = df.groupby(cols).size()
    return {"quase-identificadores": " + ".join(cols), "grupos": len(tam), "menor grupo (k)": int(tam.min()),
            "% clientes em grupos < 5": round(100 * tam[tam < 5].sum() / len(df), 1)}
dc = gold["dim_cliente"].merge(gold["fato_cliente_vendas"][["cliente_id", "educacao_id", "estado_civil_id"]], on="cliente_id")
res = pd.DataFrame([
    k_anon(silver, ["ano_nascimento", "educacao", "estado_civil", "n_filhos"]),
    k_anon(dc, ["faixa_etaria", "educacao_id", "estado_civil_id", "tem_filhos"]),
    k_anon(dc, ["faixa_etaria", "faixa_renda", "educacao_id", "tem_filhos"]),
])
res.to_csv(BASE / "catalogo" / "risco_reidentificacao.csv", index=False)
res

**Leitura (preencher após executar):** com atributos exatos (primeira linha), **[preencher]%** dos clientes estão em grupos com menos de 5 pessoas; usando faixas (linhas 2 e 3), cai para **[preencher]%**. Por isso o MVP trabalha com faixas e não expõe registros individuais. O risco prático é baixo (base pública, sem nome/contato, ID sem correspondência externa), mas não é zero, e a combinação com outras bases poderia aumentá-lo.

## 12. Autoavaliação (seção 6 do PDF)
> Preencha com a sua experiência real depois de executar tudo no Databricks. Os itens abaixo já listam as limitações conhecidas.

**1. Quais das perguntas de negócio (P1–P5) foram respondidas e quais não foram? Por quê?**
- P1: [preencher — k escolhido, silhouette e se os grupos são realmente separados]
- P2: [preencher] · P3: [preencher] · P4: [preencher — depende da premissa de Z_*] · P5: [preencher]
- Não respondidas/parcialmente: [preencher e justificar]

**2. Que limitações dos dados condicionaram os resultados?**
- Dados de ~2012–2014 (defasados); sem custo/margem por produto (receita ≠ lucro).
- Significado de `Z_CostContact` e `Z_Revenue` desconhecido: o retorno estimado de P4 é só uma ilustração sob premissa.
- `Response` refere-se a uma única campanha; taxas baixas (~15%) deixam comparações por segmento ruidosas.
- Possível viés de seleção da amostra (não se sabe como os 2.240 clientes foram escolhidos).
- Outlier de renda e idades implausíveis exigiram imputação; ver `flag_*`.

**3. Que decisões técnicas você tomaria de outra forma?**
- [preencher — ex.: testar outros algoritmos (GMM/hierárquico), outro conjunto de variáveis, ou usar PySpark MLlib; dificuldades com o Free Edition]

**4. Que extensões transformariam o MVP em solução de uso contínuo?**
- Ingestão incremental e agendada (Auto Loader/Jobs), testes de qualidade automáticos (expectations), reclusterização periódica com monitoramento de deriva, dados de custo/margem reais, teste A/B das ações do playbook, painel (Dashboard) para o time comercial.
- [preencher outras ideias]

## 13. README, LICENSE e estrutura do repositório

In [ ]:
# Estrutura final do repositório (pasta repo/) + README.md e LICENSE
import textwrap
REPO = BASE / "repo"
for sub in ["notebooks", "scripts", "catalogo", "evidencias"]:
    (REPO / sub).mkdir(parents=True, exist_ok=True)

mermaid_md = (BASE / "catalogo" / "diagrama_estrela.md").read_text().split("\n", 2)[2]
k_loc = estab["k"]

readme = textwrap.dedent(f"""\
# MVP — Clusterização de clientes e estratégia comercial

**Aluno:** {NOME_ALUNO} · **Matrícula:** {MATRICULA}
**Disciplina:** MVP — Engenharia de Produção / UnB · Prof. André Luiz Marques Serrano
**Plataforma:** Databricks Free Edition (Delta Lake, arquitetura medalhão) · **Dados:** Kaggle — Customer Personality Analysis

## 1. Objetivo
**Problema:** a empresa trata a base de clientes de forma homogênea.
**Hipótese:** existem segmentos naturais com valor, comportamento e resposta a campanhas diferentes; priorizar o orçamento comercial por segmento aumenta o retorno.

**Perguntas de negócio**
1. **P1.** Quantos segmentos naturais de clientes existem (K-Means sobre RFM + perfil demográfico) e qual o perfil de cada um em renda, composição familiar, gasto e frequência?
2. **P2.** Qual a contribuição de cada segmento para a receita total e o gasto médio por cliente? Que fração dos clientes concentra 80% do gasto (Pareto)?
3. **P3.** Como os segmentos diferem em mix de categorias (vinhos, carnes, peixes, frutas, doces, ouro) e em canal (loja, web, catálogo)? Qual o papel das visitas web e das compras com desconto?
4. **P4.** Qual a taxa de aceitação das campanhas (1 a 5 e a última) por segmento? Sob a premissa declarada custo por contato = Z_CostContact e receita por aceite = Z_Revenue, qual segmento tem melhor retorno estimado? (O significado de Z_* é uma premissa: a base não o documenta.)
5. **P5.** Quais clientes têm alto gasto histórico e recência alta (risco de inatividade) e que ação de retenção/reativação faz sentido para eles?

**Entrega de negócio:** playbook comercial com uma ação por segmento (oferta, canal, prioridade).

## 2. Dados
- **Fonte:** https://www.kaggle.com/datasets/imakash3011/customer-personality-analysis · **Licença:** {LICENCA_DADOS} (conferida em {DATA_CONFERENCIA_LICENCA})
- **Coleta:** {meta['data_coleta_utc']} (UTC) · {meta['linhas']} linhas × {meta['colunas']} colunas · {meta['tamanho_bytes']} bytes · sha256 `{meta['sha256']}`
- **Formato original:** CSV separado por TAB. Metadados em `catalogo/metadados_coleta.json`.

## 3. Arquitetura e metodologia
Fluxo: **Kaggle → Volume (Unity Catalog) → Bronze → Silver → Gold → Análise**, orquestrado por um Job do Databricks (Bronze → Silver → Gold → Análise).

| Camada | Notebook | Tabelas Delta |
|---|---|---|
| Bronze | `notebooks/01_bronze_ingestao.py` | `bronze_marketing_campaign` |
| Silver | `notebooks/02_silver_limpeza_features.py` | `silver_clientes`, `silver_regras_negocio`, `silver_constantes` |
| Gold | `notebooks/03_gold_estrela_cluster.py` | `fato_cliente_vendas`, `dim_cliente`, `dim_educacao`, `dim_estado_civil`, `dim_tempo`, `dim_segmento`, `gold_clusterizacao_*`, `gold_premissas_p4` + visões `vw_*` |
| Análise | `notebooks/04_analise_p1_p5.py` | usa as visões `vw_*` |

**Modelo:** esquema estrela (grão: um cliente). Catálogo de dados em `catalogo/catalogo_dados.md` (e `.csv`) e diagrama em `catalogo/diagrama_estrela.png`.

{mermaid_md}

**Qualidade de dados** (tabela completa em `catalogo/qualidade_por_atributo.csv` e `catalogo/qualidade_testes.csv`): nulos em `Income` (24), outlier de renda (666.666), anos de nascimento implausíveis (< 1900), categorias estranhas em `Marital_Status` (Alone/Absurd/YOLO), `2n Cycle` em `Education`, colunas constantes `Z_*`. Regras de tratamento R1–R10 em `catalogo/regras_transformacao.csv`. Evidência: `evidencias/qualidade_outliers_categorias.png`.

**Clusterização:** K-Means (padronização z-score; variáveis: recência, compras totais, log do gasto, renda, idade, nº de filhos). `k` escolhido pelo maior silhouette com k ≥ {K_MIN_ESCOLHA} (cotovelo como apoio); estabilidade avaliada com 5 sementes (ARI). Parâmetros: seed={SEED}, k de 2 a {K_MAX}.

## 4. Resultados
> Complete com os números reais após rodar o notebook `04_analise_p1_p5` (um parágrafo por pergunta, referenciando a evidência).

- **P1:** [preencher] — evidência: `evidencias/p1_cotovelo_silhouette.png`, `evidencias/p1_perfil_segmentos.png`
- **P2:** [preencher] — `evidencias/p2_receita_pareto.png`
- **P3:** [preencher] — `evidencias/p3_mix_canal.png`
- **P4:** [preencher] (premissa Z_*) — `evidencias/p4_campanhas_roi.png`
- **P5:** [preencher] — `evidencias/p5_clientes_risco.png`
- **Discussão geral e playbook comercial:** [preencher]

## 5. Ética e LGPD
Licença CC0 (a conferir), sem identificadores diretos, `ID` sequencial; risco de reidentificação medido por k-anonimato (`catalogo/risco_reidentificacao.csv`); uso de faixas em vez de valores exatos nas dimensões analíticas.

## 6. Autoavaliação
[preencher: copie a seção 12 do notebook do Colab com suas respostas]

## 7. Como reproduzir
1. Abra `notebooks/colab_mvp_unb.ipynb` no Google Colab e execute todas as células (coleta, qualidade, catálogo, geração dos notebooks Databricks).
2. No Databricks Free Edition: importe os `.py` de `notebooks/`, envie `marketing_campaign.csv` para o Volume `{CATALOG}.{SCHEMA}.{VOLUME}` e execute 01 → 02 → 03 → 04 (ou o Job).
3. Confira as tabelas em *Catalog* e preencha os resultados neste README.

## 8. Estrutura do repositório
```
README.md
LICENSE
/notebooks   (01..04 Databricks + notebook do Colab)
/scripts     (mvp_core.py)
/catalogo    (catalogo_dados.csv/.md, diagrama, qualidade, regras, metadados)
/evidencias  (prints e vídeos da execução na nuvem)
```
""")
(REPO / "README.md").write_text(readme, encoding="utf-8")

licenca = f"""MIT License (código)

Copyright (c) {dt.date.today().year} {NOME_ALUNO}

Permission is hereby granted, free of charge, to any person obtaining a copy of this software and associated
documentation files (the "Software"), to deal in the Software without restriction, including without limitation the
rights to use, copy, modify, merge, publish, distribute, sublicense, and/or sell copies of the Software, and to permit
persons to whom the Software is furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE
WARRANTIES OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT.

---
Dados: Kaggle "Customer Personality Analysis" (imakash3011) — licença registrada: {LICENCA_DADOS}
(conferida em {DATA_CONFERENCIA_LICENCA}). Fonte: https://www.kaggle.com/datasets/imakash3011/customer-personality-analysis
"""
(REPO / "LICENSE").write_text(licenca, encoding="utf-8")

# copia artefatos para a estrutura do repositório
for n in ordem_tarefas:
    shutil.copy(BASE / "databricks" / f"{n}.py", REPO / "notebooks" / f"{n}.py")
shutil.copy(BASE / "databricks" / "job_definition.json", REPO / "notebooks" / "job_definition.json")
shutil.copy(BASE / "scripts" / "mvp_core.py", REPO / "scripts" / "mvp_core.py")
for arq in (BASE / "catalogo").glob("*"):
    shutil.copy(arq, REPO / "catalogo" / arq.name)
for arq in (BASE / "evidencias").glob("*"):
    shutil.copy(arq, REPO / "evidencias" / arq.name)
print("Repositório montado em", REPO)
print(*sorted(str(p.relative_to(REPO)) for p in REPO.rglob("*") if p.is_file()), sep="\n  ")

## 14. Evidências a capturar (e onde referenciá-las no README)

In [ ]:
evid = [
 ("E1", "Volume com o CSV enviado (Catalog > Volumes)", "Databricks: Catalog", "e01_volume_csv.png", "README §2 Dados"),
 ("E2", "Tabela Delta Bronze: amostra + DESCRIBE HISTORY", "Notebook 01 (saídas das células 4 e 5)", "e02_bronze.png", "README §3 (Bronze)"),
 ("E3", "Silver: log de regras (silver_regras_negocio) e contagens", "Notebook 02 (célula 2)", "e03_silver_regras.png", "README §3 (Qualidade/tratamento)"),
 ("E4", "Catalog Explorer mostrando as tabelas bronze/silver/gold no schema", "Databricks: Catalog > schema", "e04_catalogo_tabelas.png", "README §3 (tabelas Delta persistidas)"),
 ("E5", "Esquema estrela: tabelas dim_* e fato_* com contagem de linhas", "Notebook 03 (célula 2)", "e05_estrela.png", "README §3 (Modelo)"),
 ("E6", "Escolha de k: cotovelo + silhouette + estabilidade (ARI)", "Notebook 04 (P1a)", "p1_cotovelo_silhouette.png", "README §4 P1"),
 ("E7", "Perfil dos segmentos (tabela + mapa de calor)", "Notebook 04 (P1b)", "p1_perfil_segmentos.png", "README §4 P1"),
 ("E8", "Receita por segmento e curva de Pareto", "Notebook 04 (P2)", "p2_receita_pareto.png", "README §4 P2"),
 ("E9", "Mix de categorias, canais, visitas web e desconto", "Notebook 04 (P3)", "p3_mix_canal.png", "README §4 P3"),
 ("E10", "Taxas de campanha e ROI estimado (premissa Z_*)", "Notebook 04 (P4)", "p4_campanhas_roi.png", "README §4 P4"),
 ("E11", "Lista de clientes em risco (top 20) e contagem por segmento", "Notebook 04 (P5)", "p5_clientes_risco.png", "README §4 P5"),
 ("E12", "Consulta SQL de uma visão (ex.: vw_perfil_segmento)", "Databricks: SQL Editor / notebook 03", "e12_sql_vw_perfil.png", "README §3 (visões)"),
 ("E13", "Job/Workflow com 4 tarefas encadeadas e execução bem-sucedida", "Databricks: Jobs & Pipelines > Runs", "e13_job_execucao.png", "README §3 (orquestração ETL)"),
 ("E14", "Qualidade: outliers e categorias estranhas (já gerado no Colab)", "Colab seção 5", "qualidade_outliers_categorias.png", "README §3 (Qualidade)"),
 ("E15", "Ensaio local da escolha de k (já gerado no Colab)", "Colab seção 7", "ensaio_local_escolha_k.png", "README §3 (Clusterização)"),
 ("E16", "Repositório público aberto em janela anônima", "GitHub", "e16_repo_publico.png", "Postagem no fórum"),
]
ev = pd.DataFrame(evid, columns=["id", "o que capturar", "onde", "nome sugerido do arquivo", "onde referenciar"])
ev.to_csv(BASE / "evidencias" / "lista_evidencias.csv", index=False)
tabela_md = ["| " + " | ".join(ev.columns) + " |", "|" + "---|" * len(ev.columns)]
tabela_md += ["| " + " | ".join(str(v) for v in r) + " |" for r in ev.itertuples(index=False)]
(BASE / "evidencias" / "LISTA_EVIDENCIAS.md").write_text(
    "# Evidências a capturar\n\nSalve em `evidencias/` com o nome sugerido e referencie no ponto do README que a sustenta (imagens legíveis).\n\n"
    + "\n".join(tabela_md), encoding="utf-8")
shutil.copy(BASE / "evidencias" / "LISTA_EVIDENCIAS.md", REPO / "evidencias" / "LISTA_EVIDENCIAS.md")
ev

## 15. Empacotar para download

In [ ]:
zip_path = shutil.make_archive(str(BASE / "mvp_unb_repositorio"), "zip", REPO)
print("Pacote:", zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("Fora do Colab: o arquivo .zip está na pasta de trabalho.")

### Próximos passos
1. Suba o conteúdo de `repo/` (o `.zip`) para um repositório **público** no GitHub; copie também este `.ipynb` para `/notebooks`.
2. Rode os notebooks no Databricks (seção 10), capture as evidências (seção 14) e preencha os `[preencher ...]` do notebook 04, do README e da autoavaliação.
3. Confira o link do repositório em **janela anônima** e poste no fórum da disciplina (até 2 respostas).